# I. The pipeline to analyse RNA seq data

## I.1 Cleaning sequences

### I.1.a Quality control

Raw sequencing read quality was assessed using [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/download.html#fastqc). Summary reports across all samples were generated with [MultiQC](https://docs.seqera.io/multiqc/getting_started/installation), allowing an overview of sequencing quality metrics prior to read trimming.

Raw sequencing data were initially organized in directories by species and sequencing run. For simplicity and readability in this pipeline, we refer to a single `raw_files` directory in the code.  

In [ ]:
# Define input and output directories

InputFolderPath=/PATH/raw_files   # Directory containing raw FASTQ files 
OutputFolderPath=$InputFolderPath/QC_trimming

# Move to working directory
cd $InputFolderPath

# List of sample identifiers (one per line)
# This file can be generated using the creation_sample_list .sh script
# available in the toolbox directory
FileList=samples_list.txt 

# Create output directories
mkdir -p $OutputFolderPath/
mkdir -p $OutputFolderPath/raw_QC/

# Run FastQC on raw FASTQ files
# Forward and reverse reads are processed separately

for f in $(cat $FileList);
do
	fastqc -o $OutputFolderPath/raw_QC/ $InputFolderPath/$f\_R1.fastq.gz &
done
wait
for f in $(cat $FileList);
do
	fastqc -o $OutputFolderPath/raw_QC/ $InputFolderPath/$f\_R2.fastq.gz &
done
wait

# Aggregate FastQC reports into a single summary using MultiQC
multiqc -n $OutputFolderPath/raw_QC/report_raw_fastq  $OutputFolderPath/raw_QC/ 


FastQC and MultiQC outputs were inspected to evaluate base quality scores, adapter content, and other sequencing metrics. Based on these results, trimming parameters were selected accordingly (see Table S1).

### I.1.b Trimming

Read trimming was performed using [Trim Galore](https://github.com/FelixKrueger/TrimGalore)

In [ ]:
# Trimming parameters are adjusted based on species-specific FastQC results

for f in $(cat $FileList);
do
	trim_galore --phred33 -q 20 --gzip --length 50 --cores 1 --paired --clip_R1 12 --clip_R2 12 --three_prime_clip_R1 2 --three_prime_clip_R2 2 --trim-n -o $OutputFolderPath/ $InputFolderPath/$f\__1_.fq.gz $InputFolderPath/$f\__2_.fq.gz &

done
wait

# Trim Galore options:
# --phred33                 Use Phred+33 quality score encoding
# -q 20                     Trim low-quality bases (Phred score < 20)
# --length 50               Discard reads shorter than 50 bp after trimming
# --gzip                    Compress output FASTQ files
# -o 						Output directory
# --paired                  Process paired-end reads
# --cores 1                 Run one trimming job per core
# --trim-n                  Remove Ns from read ends
#
# Species- and dataset-specific options:
# --clip_R1 / --clip_R2             				Remove bases from the 5' end of reads
# --three_prime_clip_R1 / --three_prime_clip_R2		Remove bases from the 3' end after trimming
# These parameters were selected based on FastQC results (see Table S1).

# Rename Trim Galore output files to remove _val_1 / _val_2 suffixes
for f in $OutputFolderPath/*.fq.gz; do NEWNAME=`echo $f | sed 's/_val_1//'`; mv $f $NEWNAME; done
for f in $OutputFolderPath/*.fq.gz; do NEWNAME=`echo $f | sed 's/_val_2//'`; mv $f $NEWNAME; done

For one individual (*U10_37_mother* from the *Silene latifolia* family), which was sequenced at an earlier time point, base quality scores were encoded in Phred64 rather than the standard Phred33 format. As a result, the following command was applied.

In [ ]:
trim_galore --phred64 -q 20 --gzip --length 50 --cores 1 --paired --clip_R1 12 --clip_R2 12 --three_prime_clip_R1 2 --three_prime_clip_R2 2 --trim-n -o $OutputFolderPath/ $InputFolderPath/U10_37_mother__1_.fq.gz $InputFolderPath/U10_37_mother__2_.fq.gz

Quality control was repeated on trimmed reads to verify the effectiveness of the trimming step and to ensure that quality issues observed in raw data were resolved.

In [ ]:
# Run FastQC on trimmed FASTQ files
mkdir -p $OutputFolderPath/trimmed_QC/
for f in $(cat $FileList);
do
	fastqc -o $OutputFolderPath/trimmed_QC/ $OutputFolderPath/$f\__1_.fq.gz &
done
wait
for f in $(cat $FileList);
do
	fastqc -o $OutputFolderPath/trimmed_QC/ $OutputFolderPath/$f\__2_.fq.gz &
done
wait

# Aggregate FastQC reports using MultiQC

multiqc -n $OutputFolderPath/trimmed_QC/report_trimmed_fastq  $OutputFolderPath/trimmed_QC/ 


In some cases, trimming was performed iteratively, with two rounds generally required for *Silene* samples to achieve satisfactory read quality. All trimming parameters and the number of trimming iterations per dataset are reported in Table S1.

For Artemia crosses data, trimming was done using [Cutadapt](https://doi.org/10.14806/ej.17.1.200) in two steps (first removal of adapters and filtering quality and then cutting the ends) using the following commands :

In [ ]:
#first step 
for f in $(cat $FileList);
do
    ADAPTER_FWD=AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC
    ADAPTER_REV=AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT
    cutadapt -a $ADAPTER_FWD -A $ADAPTER_REV -q 20 --minimum-length 50 --cores=2 -o $OutputFolderPath/trimmed/$f\_R1.fastq.gz -p $OutputFolderPath/trimmed/$f\_R2.fastq.gz $InputFolderPath/$f\__1_.fq.gz $InputFolderPath/$f\__2_.fq.gz &
    # --cores=2 runs each analysis on two cores (=CPU)
    # -a and -A remove adapters in both forward and reverse
    # -q 20 trims low-quality ends from reads 
    # --minimum-length 50 Discard processed reads that are shorter than 50
done

#second step 

for f in $(cat $FileList);
do

    cutadapt -u 10 -U 10 --cores=2 -o $OutputFolderPath/trimmed/trimmed_twice/$f\_R1.fastq.gz -p $OutputFolderPath/trimmed/trimmed_twice/$f\_R2.fastq.gz $InputFolderPath/trimmed/$f\_R1.fastq.gz $InputFolderPath/trimmed/$f\_R2.fastq.gz &
    # -u and -U : deleted 10 bases in each ends (3' and 5') for forward and reverse
done

### I.1.c Checking for contamination

Potential contamination from non-target species was assessed using [FastQ Screen](https://github.com/StevenWingett/FastQ-Screen), which aligns reads against multiple reference genomes and reports the proportion of reads mapping to each species, as well as ribosomal RNA content. [Bowtie2](https://github.com/BenLangmead/bowtie2) was used as the alignment backend.

FastQ Screen provides a clear visualization of contamination, but does not remove contaminating reads. In one individual (*M5_urm1230_male*), human RNA contamination was detected. Contaminating sequences were subsequently removed using [Kraken2](https://github.com/DerrickWood/kraken2).


In [ ]:
# Create output directory for FastQ Screen results
mkdir -p $OutputFolderPath/fastq_screen
time {
for f in $(cat $FileList);
do
	fastq_screen --threads 16 --outdir $OutputFolderPath/fastq_screen --aligner bowtie2 $OutputFolderPath/$f\_R1.fq.gz $OutputFolderPath/$f\_R2.fq.gz --conf /PATH/FastQ_Screen_Genomes/fastq_screen.conf

done
}
# Run FastQ Screen on trimmed reads
# --threads: number of threads
# --aligner: alignment backend (bowtie2)
# --conf: path to FastQ Screen configuration file specifying reference genomes

# Kraken command for M5_urm1230_male. The standard Kraken 2 database (containing common genomes such as human, mouse, etc.) has been downloaded and stored in the "database_conta" directory.
kraken2 --db PATH/database_conta/ --threads 50 --report test_kraken --paired $OutputFolderPath/trimmed_QC/M5_urm1230_male_R1_val_1.fq.gz $OutputFolderPath/trimmed_QC/M5_urm1230_male_R2_val_2.fq.gz



### I.1.d removal of ribosomal RNA (rRNA)

FastQ Screen revealed the presence of ribosomal RNA (rRNA) in some samples. To remove these sequences, we used [RiboDetector](https://github.com/hzi-bifo/RiboDetector). Only high-confidence non-rRNA reads were retained for downstream analyses.

In [ ]:
# Create output directory for RiboDetector results
mkdir -p $OutputFolderPath/ribodetector

for f in $(cat $FileList);
do
	ribodetector_cpu -t 14 -l 100 -i $OutputFolderPath/$f\__1_.fq.gz $OutputFolderPath/$f\__2_.fq.gz -e rrna -o $OutputFolderPath/ribodetector/$f\__1_.nonrrna.fq $OutputFolderPath/ribodetector/$f\__2_.nonrrna.fq
done
# Run RiboDetector on paired-end reads
# -t THREADS: number of threads per sample
# -l LEN: average read length
# -i: input FASTQ files (paired-end)
# -e rrna/norrna/both/none: which class of reads to output
#    norrna: keep only high-confidence non-rRNA reads
# --chunk_size: optional, for low memory usage (not needed if sufficient RAM)

# Compress non-rRNA FASTQ files
for f in $(cat $FileList);
do
	gzip $OutputFolderPath/ribodetector/$f\__1_.nonrrna.fq &
done
wait
for f in $(cat $FileList);
do
	gzip $OutputFolderPath/ribodetector/$f\__2_.nonrrna.fq &
done
wait

# Remove the ".nonrrna" suffix from output files
for f in $OutputFolderPath/ribodetector/*.fq.gz; do NEWNAME=`echo $f | sed 's/\.nonrrna//'`; mv $f $NEWNAME; done

After cleaning (quality trimming, contamination removal, rRNA removal), the resulting files were somewhat heterogeneous because not all individuals required the same processing steps. To streamline downstream analyses, all final cleaned and trimmed reads were consolidated into a single directory (`trimmed_files`). Within this directory, files remain subdivided by species and sequencing run, but for the purpose of the pipeline, all steps refer to the files as organized in `trimmed_files`.

## I.2 Mapping

To map the sequencing reads, we used [STAR](https://github.com/alexdobin/STAR/archive/2.7.11a.tar.gz), as it performs spliced alignment against the reference genome and additionally produces alignments in transcriptomic coordinates. This two-step strategy reduces background noise by retaining only reads overlapping annotated transcripts, while excluding reads mapping to non-coding or unannotated regions.

Transcript quantification was performed using [RSEM](https://github.com/deweylab/RSEM/archive/v1.3.3.tar.gz), which requires transcriptome-aligned reads generated by STAR. Read alignment and post-processing also relied on [samtools](https://github.com/samtools/samtools/releases/download/1.19/samtools-1.19.tar.bz2).


### I.2.a Genome Index and reference

First, a genome index was generated from the reference assembly. This pipeline was applied to two genera, *Artemia* and *Silene*. Because reference files and some alignment options differ between species, species-specific settings are indicated explicitly in the code below.

In [ ]:
# Reference genome (species-specific)
GenomeFile=/PATH/genome_reference/Artemia_sinica_male_genome_chromosome_level.fasta 
# S.latifolia_v4.0.genome_no_Y.fna for Silene 

# Annotation files (species-specific)
gtfFile=/PATH/genome_reference/artemia_sinica_annotation_2024.filtered.gtf # for Artemia (gtf format)
gff3File=/PATH/genome_reference/S.latifolia_v4.0.gff.gff3_polished_no_Y # for Silene (gff3 format)

# Directory containing trimmed FASTQ files
FastqInputFolder=/PATH/trimmed_files 

# Directory containing Genome Index (species-specific, created with the following code)
GenomeIndexDirectory=/PATH/GenomeIndex/Artemia 
GenomeIndexDirectory=/PATH/GenomeIndex/Silene 

#Input list file 
FastqList=input_file_list.txt
# input_file_list.txt is a tab-delimited file located in $FastqInputFolder,
# with one line per sample:
# sampleName__1_.fq.gz   sampleName__2_.fq.gz   sampleName
# This file can be generated automatically using the script creation_input_file.sh available in the toolbox directory.

# Generate the STAR genome index (performed once per reference genome)
WorkingDirectory=/PATH/genome_reference
cd $WorkingDirectory

mkdir -p $GenomeIndexDirectory
time {
STAR --runMode genomeGenerate --limitGenomeGenerateRAM 25000000000 --genomeDir $GenomeIndexDirectory --genomeFastaFiles $GenomeFile --runThreadN 25 --sjdbGTFfile $gtfFile --sjdbGTFtagExonParentTranscript Parent
}
wait
# --runMode genomeGenerate option directs STAR to run genome indices generation job.
# --limitGenomeGenerateRAM 25000000000 limits RAM usage to 25Gb (check how much RAM you have first)
# --genomeDir specifies path to the directory where the genome indices are stored.
# --genomeFastaFiles specifies FASTA file with the genome reference sequences.
# --sjdbGTFfile specifies the path to the file with annotated transcripts in the standard GTF format. STAR will extract splice junctions from this file and use them to greatly improve accuracy of the mapping.
# --runThreadN 25 number of threads to adapt
# --sjdbGTFtagExonParentTranscript Parent is a necessary option for gff3 files to extract exons
# Notes : 
# for Silene, --sjdbGTFfile $gff3File is used instead of --sjdbGTFfile $gtfFile by 

# Extract the transcriptome sequence using RSEM (performed once per reference genome)
rsem-prepare-reference --gtf $gtfFile $GenomeFile reference 
# For Silene, use: --gtf $gff3File

# RSEM produces multiple output files; only the transcript FASTA is used downstream
mv reference.transcripts.fa reference.fasta
rm reference.chrlist reference.n2g.idx.fa reference.grp reference.gtf reference.idx.fa reference.seq reference.ti


**Notes** : We modified the original *Artemia* annotation by keeping the longest transcript for each gene (not needed for *Silene* where there is only one transcript). Here is the R code we used to do so  

In [ ]:
library(tidyverse)
file_gtf_artemia <- "PATH/genome_reference/artemia_sinica_annotation_2024.gtf"

gtf_function <- function(file_gtf) {
  
  data_gtf <- read.table(file_gtf, sep="\t", header=FALSE, comment.char="#", stringsAsFactors=FALSE)
  colnames(data_gtf) <- c("Chromosome", "Source", "Type", "Start", "End", "Score", "Strand", "Phase", "Attributes")
  data_gtf <- data_gtf %>% separate(col = Attributes, 
           into = c("gene_ID", "transcript_ID", "ID", "Parent"),
           sep = ";",
           convert = TRUE, 
           extra = "drop") 
  data_gtf$ID <- gsub("ID ", "", data_gtf$ID)
  data_gtf$gene_ID <- gsub("gene_id ", "", data_gtf$gene_ID)
  data_gtf$transcript_ID <- gsub("transcript_id ", "", data_gtf$transcript_ID)
  data_gtf$Parent <- gsub("Parent ", "", data_gtf$Parent)
  data_gtf <- data_gtf %>%
  mutate(ID = str_trim(ID)) %>% 
  filter(startsWith(Chromosome,"Chromosome")) #remove scaffolds and contigs that are not assigned to a chromosome
  return(data_gtf)
  }

data_gtf_artemia<-gtf_function(file_gtf_artemia)

data_gtf_artemia_total <- data_gtf_artemia %>% filter(Type=="transcript")
data_gtf_artemia_longest_transcript <- data_gtf_artemia_total %>% mutate(Length=End-Start+1) %>%
    group_by(gene_ID) %>%
    filter(Length == max(Length)) %>% #to keep only the longest transcript
    slice_sample(n = 1) %>%  #if two transcripts have the same length, chose randomly
    ungroup()
data_gtf_artemia_transcripts_supprimes <- data_gtf_artemia_total %>%
  anti_join(data_gtf_artemia_longest_transcript, by = "transcript_ID")
write_csv(tibble(data_gtf_artemia_transcripts_supprimes$transcript_ID),"PATH/genome_reference/Artemia_transcript_to_remove.csv")


### I.2.b Mapping

In [ ]:
# Working directory for read alignments.
# This directory can be defined freely; in our case, separate directories were used for different run.
WorkingDirectory=/PATH/Alignments/runX 

mkdir -p $WorkingDirectory/AlignedReads

time {
while read forward reverse ind 
do
	STAR --genomeDir $GenomeIndexDirectory --sjdbGTFfile $gtfFile --readFilesIn $FastqInputFolder/$forward $FastqInputFolder/$reverse --readFilesCommand gunzip -c --runThreadN 20 --outSAMtype BAM SortedByCoordinate --quantMode TranscriptomeSAM --quantTranscriptomeSAMoutput BanSingleEnd --outMultimapperOrder Random --outFileNamePrefix $WorkingDirectory/AlignedReads/$ind
done < $FastqInputFolder/$FastqList

}

# STAR alignment options:
# --genomeDir specifies the directory containing the STAR genome index.
# --sjdbGTFfile provides the transcript annotation used to improve splice junction detection
#   (for Silene latifolia, use the GFF3 annotation file instead of the GTF).
# --readFilesCommand gunzip -c allows reading gzipped FASTQ files.
# --outSAMtype BAM SortedByCoordinate outputs coordinate-sorted BAM files.
# --quantMode TranscriptomeSAM outputs alignments in transcript coordinates
#   (Aligned.toTranscriptome.out.bam), required for downstream transcript quantification.
# --outFileNamePrefix sets the output basename (here, the sample identifier).
# --outMultimapperOrder Random assigns multi-mapping reads randomly.


# Move transcriptome-aligned BAM files to the working directory
mv $WorkingDirectory/AlignedReads/*Aligned.toTranscriptome.out.bam $WorkingDirectory/

# Rename transcriptome BAM files by removing STAR-specific suffixes
for f in *Aligned.toTranscriptome.out.bam; do NEWNAME=`echo $f | sed 's/Aligned\.toTranscriptome\.out//'`; mv $f $NEWNAME; done


### I.2.c Sorting and check outputs

Mapped reads can be inspected using [samtools](http://www.htslib.org/doc/samtools.html) and visualized with [BamView](http://sanger-pathogens.github.io/Artemis/BamView/).

Before visualization or downstream analyses, BAM files were filtered and sorted. The `-F 4` option was used to exclude unmapped reads, while the `-F 0x100` option was applied to remove secondary alignments. In cases where a read mapped with equal probability to multiple genomic locations, only the primary alignment (randomly assigned by STAR) was retained.

In [ ]:
# Sort BAM files and retain only primary mapped reads.
# Unmapped reads (-F 4) and secondary alignments (-F 0x100) are excluded.
# Sorting is performed using 16 threads and 2 GB of RAM per thread.
# An index is generated for each sorted BAM file.
time {
while read forward reverse ind 
do
	samtools view -F 4  -F 0x100 -h -@ 16 -b ${ind}.bam | samtools sort -m 2G -@ 16 -o ${ind}\_sorted.bam ;
	samtools index -@ 16 ${ind}\_sorted.bam ;
done < $FastqInputFolder/$FastqList
}

# Extract read counts per contig
time { 
for f in $(ls *sorted.bam);
do
	samtools idxstats $f > $f.idxstats ;
done
}
# The idxstats output contains the following columns:
# contig   contig_length   number_mapped_reads   number_unmapped_reads

# Compute global alignment statistics for each sample
time {
for f in $(ls *sorted.bam);
do
    samtools flagstat $f > $f.flagstat ;
done
}

### I.2.d Merging bam files

In some cases, two or more sequencing runs were generated for the same individual. This may occur when an initial run did not meet quality standards or when technical issues arose during sequencing, leading the sequencing facility to perform an additional run.

When multiple runs are available for the same individual, the corresponding BAM files were merged into a single file prior to downstream analyses. The merged BAM file was then re-sorted to ensure proper ordering of alignments.

In [ ]:
# Path to the directory containing multiple sequencing runs for the same samples
CommunPath=/PATH/Alignments

# List of sample identifiers to merge.
# This list can be reused from the mapping step (e.g. from one sequencing run).
# If one run does not contain all individuals, using its sample list avoids
# attempting to merge non-existing BAM files.
FileList=/PATH/Alignments/run1/samples_list.txt 

# Create a directory to store merged BAM files
mkdir -p $CommunPath/merged 
cd $CommunPath/merged

# Merge BAM files originating from multiple sequencing runs for the same individual.
# Merging is followed by sorting to ensure proper coordinate ordering.
# Operations are performed using 20 threads and 2 GB of RAM per thread.

time {
for f in $(cat $FileList);
do
	samtools merge -o $f\_merged.bam -@ 20 $CommunPath/run1/$f\_sorted.bam $CommunPath/run2/$f\_sorted.bam;
	samtools sort -o $f\_merged_sorted.bam -m 2G -@ 20 $f\_merged.bam ;
done
}

## I.3 Extracting genotypes and XY sequences

### I.3.a File organization after mapping

After mapping reads to the reference genomes, the output files were heterogeneous because some individuals were sequenced in multiple runs and needed to be merged, while others only had a single run.  

To simplify downstream analyses, we reorganized the `Alignments` directory as follows:  

- One subdirectory per species/family (e.g., `Silene_latifolia`, `Artemia_sinica`) for family-level data.  
- One subdirectory per population-level dataset, if applicable (e.g., `Population_data/Silene_latifolia`).  
- Within each of these directories, all BAM files for each individual are stored, including merged runs if applicable.  

This structure allows the pipeline to reference individual BAM files in a consistent way, while retaining the separation between families and populations as needed for analyses.

**Note:** In the following steps of the pipeline, instead of explicitly writing the species or population directory, we will use the placeholder `focal_species` to refer to the relevant subdirectory.

<pre>
PATH
├── raw_files
    ├── run1
        ├── Artemia_sinica
        └── ...
    ├── run2
    └── run3
├── trimmed_files
    ├── run1
        ├── Artemia_sinica
        └── ...
    ├── run2
    └── run3
└── Alignments
    ├── run1
    ├── run2
    ├── run3
    ├── merged
    ├── Artemia_sinica
    ├── Artemia_urmiana
    ├── Artemia_franciscana
    ├── Silene_diclinis
    ├── Silene_dioica
    ├── Silene_heuffelii
    ├── Silene_latifolia
    ├── Silene_marizii
    ├── Silene_viscosa
    └── Population_data
        ├── Silene_diclinis
        ├── Silene_dioica
        ├── Silene_heuffelii
        ├── Silene_latifolia
        └── Silene_marizii
</pre>


### I.3.b Shortcuts

We used `reads2snp_contam.bin` (available in our tool_box directory) for genotype extraction.  

To simplify repeated analyses, variables were created for each individual in each species. These variables are primarily for clarity and reproducibility within the code, and help track which BAM file corresponds to each sample.

#### For *Artemia*

In [ ]:
# Artemia sinica
Species=Artemia_sinica
FamilyFile=Artemia_sinica_Family
mother=M_sinica1206_mother_sorted
father=P_sinica1206_father_sorted
female1=F1_sinica1206_female_sorted
female2=F2_sinica1206_female_sorted
female3=F3_sinica1206_female_sorted
female4=F4_sinica1206_female_sorted
female5=F5_sinica1206_female_sorted
male1=M1_sinica1206_male_sorted
male2=M2_sinica1206_male_sorted
male3=M3_sinica1206_male_sorted
male4=M4_sinica1206_male_sorted
male5=M5_sinica1206_male_sorted

# Artemia urmiana
Species=Artemia_urmiana
FamilyFile=Artemia_urmiana_Family
mother=M_urm1230_mother_sorted
father=P_urm1230_father_sorted
female1=F1_urm1230_female_sorted
female2=F2_urm1230_female_sorted
female3=F3_urm1230_female_sorted
female4=F4_urm1230_female_sorted
female5=F5_urm1230_female_sorted
male1=M1_urm1230_male_sorted
male2=M2_urm1230_male_sorted
male3=M3_urm1230_male_sorted
male4=M4_urm1230_male_sorted
male5=M5_urm1230_male_kraken_sorted # cleaned for human RNA contamination

# Artemia franciscana (outgroup)
Species=Artemia_franciscana
FamilyFile=Artemia_franciscana

#### For *Silene*

In [ ]:
#Silene diclinis
Species=Silene_diclinis
FamilyFile=Silene_diclinis_Family
mother=Sd_3_Biu_Mother_merged_sorted
father=Sd_9_Qua_Father_merged_sorted
female1=Sd_1_female_merged_sorted
female2=Sd_2_female_merged_sorted
female3=Sd_4_female_merged_sorted
female4=Sd_5_female_merged_sorted
female5=Sd_6_female_merged_sorted
male1=Sd_10_male_merged_sorted
male2=Sd_11_male_merged_sorted
male3=Sd_12_male_merged_sorted
male4=Sd_14_male_merged_sorted
male5=Sd_15_male_merged_sorted

#Silene dioica
Species=Silene_dioica
FamilyFile=Silene_dioica_Family
mother=SD2_NORA16_mother_merged_sorted
father=SD2_NL3_father_merged_sorted
female1=SD2_10_female_merged_sorted
female2=SD2_11_female_merged_sorted
female3=SD2_13_female_merged_sorted
female4=SD2_18_female_merged_sorted
female5=SD2_20_female_merged_sorted
male1=SD2_2_male_merged_sorted
male2=SD2_7_male_merged_sorted
male3=SD2_12_male_merged_sorted
male4=SD2_14_male_merged_sorted
male5=SD2_19_male_merged_sorted

#Silene heuffelii
Species=Silene_heuffelii
FamilyFile=Silene_heuffelii_Family
mother=SH_BUL67_Mother_merged_sorted
father=SH_ROM_M3_Father_merged_sorted
female1=SH_1_female_merged_sorted
female2=SH_4_female_merged_sorted
female3=SH_5_female_merged_sorted
female4=SH_6_female_merged_sorted
female5=SH_8_female_merged_sorted
male1=SH_10_male_merged_sorted
male2=SH_11_male_merged_sorted
male3=SH_13_male_merged_sorted
male4=SH_16_male_merged_sorted
male5=SH_9_male_merged_sorted

#Silene marizii
Species=Silene_marizii
FamilyFile=Silene_marizii_Family
mother=SM_Gov_3_Mother_merged_sorted
father=SM_Mag_1_Father_merged_sorted
female1=SM_1_female_merged_sorted
female2=SM_2_female_merged_sorted
female3=SM_6_female_merged_sorted
female4=SM_7_female_merged_sorted
female5=SM_9_female_merged_sorted
male1=SM_4_male_merged_sorted
male2=SM_5_male_merged_sorted
male3=SM_8_male_merged_sorted
male4=SM_10_male_merged_sorted
male5=SM_13_male_merged_sorted

#Silene latifolia
Species=Silene_latifolia
FamilyFile=Silene_latifolia_Family
mother=U10_37_mother_sorted
father=Leuk144-3_father_sorted
female1=C1_26_female_sorted
female2=C1_27_female_sorted
female3=C1_29_female_sorted
female4=C1_34_female_sorted
male1=C1_01_male_sorted
male2=C1_04_male_sorted
male3=C1_05_male_sorted
male4=C1_3_male_sorted

#Silene viscosa (outgroup)
Species=Silene_viscosa
FamilyFile=Silene_viscosa


### I.3.c Extracting genotypes

In [ ]:
# Set the working directory to the focal species within the Alignments folder
WorkingDirectory=/PATH/Alignments/focal_species 
cd $WorkingDirectory

# Create a list of all sorted BAM files to use as input for reads2snp
ls *sorted.bam > bam_list.txt

# Run reads2snp for genotype inference
time {
reads2snp_contam.bin -acc -nbth 20 -aeb -par 0 -bamlist bam_list.txt -bamref /PATH/genome_reference/reference.fasta -bqt 20 -rqt 0 -min 3 -out ${FamilyFile}
}
# Key options:
# -acc      : accept complex count patterns
# -nbth 20  : use 20 threads for parallel processing
# -aeb      : account for allelic expression bias
# -par 0    : disable paralog filtering (important for sex chromosome analyses, 
#             as X/Y SNPs may appear as paralogs)
# -bamlist  : input BAM files (bam_list.txt)
# -bamref   : reference transcriptome (reference.fasta)
# -bqt 20   : only consider bases with Phred quality ≥ 20
# -rqt 0    : do not filter on mapping quality (Y-linked reads may map less confidently than X)
# -min 3    : require at least 3 reads to call a genotype
# -out      : base name for output files (FamilyFile)

# take a look at the output files:
head ${FamilyFile}.alr
head ${FamilyFile}.gen
head ${FamilyFile}.vcf

### I.3.b Running SEX-DETector to infer X and Y sequences

To infer the sex-determination system for each species or family, multiple versions of [SEX-DETector](https://gitlab.in2p3.fr/sex-det-family/sex-detector-plusplus) were tested.  
Different inheritance models were evaluated (autosomal, XY, ZW), and all available individuals were included in each analysis (typically 4 or 5 daughters and sons per family).  

For species with fewer offspring (e.g., *Silene latifolia*), the commands were adapted to the number of available individuals.

**Note on command options** (applies to all models unless otherwise stated):  
- `--fo`: female offspring  
- `--mo`: male offspring  
- `--fp`: mother  
- `--mp`: father  


#### I.3.b.i Standard version 

In [ ]:
#Run model assuming no sex chromosomes (autosomal)
sex_detector_plusplus_standard ${FamilyFile}.gen -m autosomal -o ${FamilyFile}_plusplus_no_sex_chr -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait
# Run model assuming an XY system
sex_detector_plusplus_standard ${FamilyFile}.gen -m xy -o ${FamilyFile}_plusplus_XY -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait
# Run model assuming a ZW system
sex_detector_plusplus_standard ${FamilyFile}.gen -m zw -o ${FamilyFile}_plusplus_ZW -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}

# --- Special case: Silene latifolia ---
# Only four daughters and four sons are available, commands adapted accordingly
sex_detector_plusplus_standard ${FamilyFile}.gen -m autosomal -o ${FamilyFile}_plusplus_no_sex_chr -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}
wait
sex_detector_plusplus_standard ${FamilyFile}.gen -m xy -o ${FamilyFile}_plusplus_XY -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}
wait
sex_detector_plusplus_standard ${FamilyFile}.gen -m zw -o ${FamilyFile}_plusplus_ZW -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}

# Compare model fit using BIC
# The model with the lowest BIC indicates the best-fitting inheritance model
tail -n 1 ${FamilyFile}_plusplus_no_sex_chr_likelihood | cut -f5
tail -n 1 ${FamilyFile}_plusplus_XY_likelihood | cut -f5
tail -n 1 ${FamilyFile}_plusplus_ZW_likelihood | cut -f5

# Check the final values of the Y genotyping errors
# If the value exceeds 0.05, a different SEX-DETector version (with homogeneous or heterogeneous error models) should be considered
tail -1 ${FamilyFile}_plusplus_XY_parameters | cut -f5,6

# Count how many genes were inferred as sex-linked and autosomal 
cat ${FamilyFile}_plusplus_XY_assignment | grep -w 'sex-linked' -c
cat ${FamilyFile}_plusplus_XY_assignment | grep -w 'autosomal' -c


For *Silene*, the Y genotyping error values were below 0.05 for the best-fitting model (XY, as expected), so no alternative SEX-Detector versions were necessary.  

For *Artemia*, although a ZW system would be expected, the XY model was consistently identified as the best-fitting system. However, only a few sex-linked genes were inferred and the Y genotyping error exceeded the threshold (0.05) by more than tenfold. Consequently, two additional SEX-Detector versions were run to account for this high error rate.

#### I.3.b.ii Heterogeneous version 

In [ ]:
#Run model assuming no sex chromosomes (autosomal)
sex_detector_heterogeneous ${FamilyFile}.gen -m autosomal -o ${FamilyFile}_hetero_no_sex_chr -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait
# Run model assuming an XY system
sex_detector_heterogeneous ${FamilyFile}.gen -m xy -o ${FamilyFile}_hetero_XY -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait
# Run model assuming a ZW system
sex_detector_heterogeneous ${FamilyFile}.gen -m zw -o ${FamilyFile}_hetero_ZW -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait

# --- Special case: Silene latifolia ---
# Only four daughters and four sons are available, commands adapted accordingly
sex_detector_heterogeneous ${FamilyFile}.gen -m autosomal -o ${FamilyFile}_hetero_no_sex_chr -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}
wait
sex_detector_heterogeneous ${FamilyFile}.gen -m xy -o ${FamilyFile}_hetero_XY -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}
wait
sex_detector_heterogeneous ${FamilyFile}.gen -m zw -o ${FamilyFile}_hetero_ZW -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}

# Compare model fit using BIC
tail -n 1 ${FamilyFile}_hetero_no_sex_chr_likelihood | cut -f5
tail -n 1 ${FamilyFile}_hetero_XY_likelihood | cut -f5
tail -n 1 ${FamilyFile}_hetero_ZW_likelihood | cut -f5

# Check the final values of the Y genotyping errors
tail -1 ${FamilyFile}_hetero_XY_parameters | cut -f5,6,7

# Count how many genes were inferred as sex-linked and autosomal 
cat ${FamilyFile}_hetero_XY_assignment | grep -w 'sex-linked' -c
cat ${FamilyFile}_hetero_XY_assignment | grep -w 'autosomal' -c


#### I.3.b.iii Homogeneous version 

In [ ]:
#Run model assuming no sex chromosomes (autosomal) 
sex_detector_homogeneous ${FamilyFile}.gen -m autosomal -o ${FamilyFile}_homo_no_sex_chr -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait
# Run model assuming an XY system
sex_detector_homogeneous ${FamilyFile}.gen -m xy -o ${FamilyFile}_homo_XY -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait
# Run model assuming a ZW system
sex_detector_homogeneous ${FamilyFile}.gen -m zw -o ${FamilyFile}_homo_ZW -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4},${female5} --mo ${male1},${male2},${male3},${male4},${male5} --fp ${mother} --mp ${father}
wait

# --- Special case: Silene latifolia ---
# Only four daughters and four sons are available, commands adapted accordingly
sex_detector_homogeneous ${FamilyFile}.gen -m autosomal -o ${FamilyFile}_homo_no_sex_chr -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}
wait
sex_detector_homogeneous ${FamilyFile}.gen -m xy -o ${FamilyFile}_homo_XY -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}
wait
sex_detector_homogeneous ${FamilyFile}.gen -m zw -o ${FamilyFile}_homo_ZW -e ${FamilyFile}.alr --fo ${female1},${female2},${female3},${female4} --mo ${male1},${male2},${male3},${male4} --fp ${mother} --mp ${father}

# Compare model fit using BIC
tail -n 1 ${FamilyFile}_homo_no_sex_chr_likelihood | cut -f5
tail -n 1 ${FamilyFile}_homo_XY_likelihood | cut -f5
tail -n 1 ${FamilyFile}_homo_ZW_likelihood | cut -f5

# Check the final values of the Y genotyping errors
tail -1 ${FamilyFile}_homo_XY_parameters | cut -f5

# Count how many genes were inferred as sex-linked and autosomal
cat ${FamilyFile}_homo_XY_assignment | grep -w 'sex-linked' -c
cat ${FamilyFile}_homo_XY_assignment | grep -w 'autosomal' -c


#### I.3.b.iv Perl version 

For downstream analyses, the Perl implementation of SEX-Detector (the original and considerably slower version) was required for all species, as several output files needed later in the pipeline are not generated by the C++ version. The Perl version was therefore run only once, without parameter re-estimation, using the parameter values inferred from the best-fitting C++ model (XY for *Silene* and ZW for *Artemia*), solely to generate the missing output files.

For *Artemia*, although BIC values indicated that the heterogeneous ZW model should be preferred, a different strategy was adopted. Our primary objective was to maximize the number of inferred sex-linked genes. In addition, results from the standard C++ version of SEX_DETector showed that the XY model was favored only because of extremely inflated Y genotyping error rates, rather than genuine support for an XY system. Consequently, parameters from the standard ZW model were retained for the Perl run, despite the BIC ranking, as this configuration yielded a higher number of sex-linked genes than the heterogeneous and homogeneous model.


In [ ]:
# Run perl model assuming an XY system for Silene
time {
tail -n 1 ${FamilyFile}_plusplus_XY_parameters > final_parameters_SEX-DETector_plusplus
SEX-DETector_prepare_file.pl ${FamilyFile}.gen ${FamilyFile}.gen_summary -hom ${female1},${female2},${female3},${female4},${female5} -het ${male1},${male2},${male3},${male4},${male5} -hom_par ${mother} -het_par ${father}
SEX-DETector.pl -skip_opt -param final_parameters_SEX-DETector_plusplus -alr ${FamilyFile}.alr -alr_gen ${FamilyFile}.gen -alr_gen_summary ${FamilyFile}.gen_summary -out ${FamilyFile}_perl -hom ${female1},${female2},${female3},${female4},${female5} -het ${male1},${male2},${male3},${male4},${male5} -hom_par ${mother} -het_par ${father} -system xy -seq -detail -detail-sex-linked -L -SEM -thr 0.8 
}
# Run perl model assuming an XY system for Silene latifolia (Only four daughters and four sons are available, commands adapted accordingly)
time {
tail -n 1 ${FamilyFile}_plusplus_XY_parameters > final_parameters_SEX-DETector_plusplus
SEX-DETector_prepare_file.pl ${FamilyFile}.gen ${FamilyFile}.gen_summary -hom ${female1},${female2},${female3},${female4} -het ${male1},${male2},${male3},${male4} -hom_par ${mother} -het_par ${father}
SEX-DETector.pl -skip_opt -param final_parameters_SEX-DETector_plusplus -alr ${FamilyFile}.alr -alr_gen ${FamilyFile}.gen -alr_gen_summary ${FamilyFile}.gen_summary -out ${FamilyFile}_perl -hom ${female1},${female2},${female3},${female4} -het ${male1},${male2},${male3},${male4} -hom_par ${mother} -het_par ${father} -system xy -seq -detail -detail-sex-linked -L -SEM -thr 0.8 
}

# Run perl model assuming an ZW system for Artemia
time {
tail -n 1 ${FamilyFile}_plusplus_ZW_parameters > final_parameters_SEX-DETector_plusplus
SEX-DETector_prepare_file.pl ${FamilyFile}.gen ${FamilyFile}.gen_summary -hom ${male1},${male2},${male3},${male4},${male5} -het ${female1},${female2},${female3},${female4},${female5} -hom_par ${father} -het_par ${mother}
SEX-DETector.pl -skip_opt -param final_parameters_SEX-DETector_plusplus -alr ${FamilyFile}.alr -alr_gen ${FamilyFile}.gen -alr_gen_summary ${FamilyFile}.gen_summary -out ${FamilyFile}_perl -hom ${male1},${male2},${male3},${male4},${male5} -het ${female1},${female2},${female3},${female4},${female5}  -hom_par ${father} -het_par  ${mother} -system zw -seq -detail -detail-sex-linked -L -SEM -thr 0.8 
}

# Compute BIC
tail -n 1 ${FamilyFile}_perl_parameters.txt | cut -f38

# Check the final values of the Y genotyping errors
tail -1 ${FamilyFile}_perl_parameters.txt | cut -f5,6

# Count how many genes were inferred as sex-linked and autosomal
cat ${FamilyFile}_perl_assignment.txt | grep -w 'sex-linked' -c
cat ${FamilyFile}_perl_assignment.txt | grep -w 'autosomal' -c

### I.3.c Sequence extraction 

#### Artemia 

In [ ]:
# Convert X/Y labels to Z/W in SEX-DETector outputs for Artemia
# (required for downstream ZW-based sequence extraction)
sed -i 's/Y/W/g' ${FamilyFile}_perl_SNPs_detail.txt
sed -i 's/X/Z/g' ${FamilyFile}_perl_SNPs_detail.txt

# Create a directory to store sequences extracted with wxyz_genotyper
mkdir -p perl_sequence_extraction

# Format SEX-DETector Perl outputs for sequence extraction with wxyz_genotyper
SEX-DETector_perl_sequences_preparation_ZW_no_filter.R /PATH/Alignements/${Species}/ ${FamilyFile}_perl_assignment.txt ${FamilyFile}_perl_SNPs_detail.txt perl_sequence_extraction/${FamilyFile}_perl_allele_freq.txt

cd perl_sequence_extraction

# Extract Z and W sequences using wxyz_genotyper
wxyz_genotyper ${FamilyFile}_perl_allele_freq.txt ../${FamilyFile}.vcf v ZW_sequences.fasta 0.8 0.95 0.6 prob_sex_linked ZW mean 

# Post-process extracted sequences:
# - convert residual X/Y labels
# - replace ambiguous bases by N
# - standardize sequence naming
sed -i 's/X/N/g' ZW_sequences.fasta
sed -i 's/\_N/\_Z/g' ZW_sequences.fasta # Restore potential _Z gene labels that were converted to _N in the previous step
sed -i 's/Y/W/g' ZW_sequences.fasta
sed -i 's/n/N/g' ZW_sequences.fasta

# Record and count the number of ZW, hemizygous, sex-linked, and "mystere" genes
# "Mystere" genes are inferred as sex-linked but lack extracted Z or W sequences,
# potentially indicating data quality issues or model misfit.
# Ideally, few or no mystere genes should be observed.
cat ZW_sequences.fasta | grep '>' | awk '{print $1}'| sed 's/>//' | grep '_Z' | sed 's/_Z//' > list_gene_ZW_perl.csv
cd ..
cat ${FamilyFile}_perl_assignment.txt | grep -w 'sex-linked' | awk '$9==0' | awk '{print $1}' > list_gene_hemi_perl.csv
cat ${FamilyFile}_perl_assignment.txt | grep -w 'sex-linked' | awk '{print $1}' > list_gene_sex_linked_perl.csv
grep -v -F -f <(sort list_gene_hemi_perl.csv) <(sort list_gene_sex_linked_perl.csv) | grep -v -F -f <(sort list_gene_ZW_perl.csv) > list_gene_mystere_perl.csv
wc -l list_gene_hemi_perl.csv 
wc -l list_gene_sex_linked_perl.csv 
wc -l list_gene_ZW_perl.csv
wc -l list_gene_mystere_perl.csv # should ideally be zero

# If any, inspect mystere genes in the SEX-DETector assignment file
cat ${FamilyFile}_perl_assignment.txt | grep -f list_gene_mystere_perl.csv 

#### Silene 

In [ ]:
# Create a directory to store sequences extracted with wxyz_genotyper
mkdir -p perl_sequence_extraction

# Format SEX-DETector Perl outputs for sequence extraction with wxyz_genotyper
SEX-DETector_perl_sequences_preparation_no_filter.R /PATH/Alignements/${Species}/merged/ ${FamilyFile}_perl_assignment.txt ${FamilyFile}_perl_SNPs_detail.txt perl_sequence_extraction/${FamilyFile}_perl_allele_freq.txt

cd perl_sequence_extraction

# Extract X and Y sequences using wxyz_genotyper
wxyz_genotyper ${FamilyFile}_perl_allele_freq.txt ../${FamilyFile}.vcf v XY_sequences.fasta 0.8 0.95 0.6 prob_sex_linked XY mean 

# Post-process extracted sequences:
# - replace ambiguous bases by N
# - standardize sequence naming
sed -i 's/X/N/g' XY_sequences.fasta
sed -i 's/\_N/\_X/g' XY_sequences.fasta # Restore potential _X gene labels that were converted to _N in the previous step
sed -i 's/n/N/g' XY_sequences.fasta

# Record and count the number of XY, hemizygous, sex-linked, and "mystere" genes
# "Mystere" genes are inferred as sex-linked but lack extracted X or Y sequences,
# potentially indicating data quality issues or model misfit.
# Ideally, few or no mystere genes should be observed.
cat XY_sequences.fasta | grep '>' | awk '{print $1}'| sed 's/>//' | grep '_X' | sed 's/_X//' > list_gene_XY_perl.csv
cd ..
cat ${FamilyFile}_perl_assignment.txt | grep -w 'sex-linked' | awk '$9==0' | awk '{print $1}' > list_gene_hemi_perl.csv
cat ${FamilyFile}_perl_assignment.txt | grep -w 'sex-linked' | awk '{print $1}' > list_gene_sex_linked_perl.csv
grep -v -F -f <(sort list_gene_hemi_perl.csv) <(sort list_gene_sex_linked_perl.csv) | grep -v -F -f <(sort list_gene_XY_perl.csv) > list_gene_mystere_perl.csv

wc -l list_gene_hemi_perl.csv 
wc -l list_gene_sex_linked_perl.csv 
wc -l list_gene_XY_perl.csv
wc -l list_gene_mystere_perl.csv

# If any, inspect mystere genes in the SEX-DETector assignment file
cat ${FamilyFile}_perl_assignment.txt | grep -f list_gene_mystere_perl.csv 

## I.4 Filtering SNPs with population data (only for *Silene*)

To further clean the data, we also used population-level datasets for *Silene*. The idea is simple: a SNP inferred as Y-linked in a family is likely a false positive if the same SNP is found in females from other populations of the same species.

Population data were processed using the same pipeline as family data (RNA-seq cleaning, mapping, and genotype extraction). For each species, we also used the list of sex-linked genes inferred from the family analyses (e.g. list_gene_sex_XY_perl.csv).

Using the script gene_XY_gen_maker.sh, the genotype file produced by reads2snp was split into one file per gene. This step is computationally intensive and usually took several hours. These per-gene genotype files were then analyzed with SNP_filter_perl.R, which identifies SNPs that should be removed because they are present in females from population data.

After this filtering step, family-level sequences were re-extracted using a dedicated sequence preparation script, as described below.

In [ ]:
# Move to the Population_data directory to split the genotype file by gene
cd /PATH/Alignments/Population_data/focal_species

gene_XY_gen_maker.sh /PATH/Alignments/${Species}/list_gene_XY_perl.csv ${FamilyFile}.gen

# Move back to the Species directory for the next steps
cd /PATH/Alignments/${Species}

# Filter false XY SNPs 
Snp_filter_perl.R ${FamilyFile}_perl_SNPs_detail.txt list_gene_XY_perl.csv /PATH/Alignments/Population_data/${Species}/gene_selection

# Create a directory to store filtered sequences and extract sequences as in the previous step
mkdir -p perl_sequence_extraction_filtered
SEX-DETector_perl_sequences_preparation_filter.R /PATH/Alignments/${Species} ${FamilyFile}_perl_assignment.txt ${FamilyFile}_perl_SNPs_detail.txt snp_to_remove.csv perl_sequence_extraction_filtered/${FamilyFile}_perl_allele_freq.txt

cd perl_sequence_extraction_filtered

wxyz_genotyper ${FamilyFile}_perl_allele_freq.txt ../${FamilyFile}.vcf v XY_sequences.fasta 0.8 0.95 0.6 prob_sex_linked XY mean 

sed -i 's/X/N/g' XY_sequences.fasta
sed -i 's/\_N/\_X/g' XY_sequences.fasta
sed -i 's/n/N/g' XY_sequences.fasta

## I.5 Compute dS 

### I.5.a. Preparing sequences for dS computation

#### *Silene* : UTR trimming and sequence checks 

Before computing dS, UTR regions must be removed from the coding sequences. A set of scripts was used for this step:

- `multiple_fasta_file.sh` to split the multi-FASTA file into one FASTA file per gene  
- `cut_UTR.R` to remove UTR regions based on annotation  
- `check_cutted_fasta.sh` to verify that trimming was correctly performed  

The final check ensures that each sequence:
- has a length divisible by three,
- starts with a start codon,
- ends with a stop codon.

This step was only required for *Silene*. In *Artemia*, no UTRs were described in the annotation, so only exonic sequences were extracted and no additional trimming was necessary.

In [ ]:
cd /PATH/Alignments/focal_species/perl_sequence_extraction_filtered/

# Split the multi-FASTA file into one FASTA file per gene
multiple_fasta_file.sh XY_sequences.fasta
wait

# Remove UTR regions from each gene sequence
cut_UTR.R fasta_to_cut/
wait
cd fasta_to_cut/

# Check that UTR trimming was successful and that sequences are valid coding sequences
check_cutted_fasta.sh cutted_fasta/
wait
# Identify and count sequences that failed the validation checks
grep 'FALSE' cut_check 
grep 'FALSE' cut_check | wc -l 


After UTR trimming, sequence integrity was checked for all five *Silene* species. For a small number of genes (one or two depending on the species), the resulting coding sequence was not divisible by three. Inspection of the alignments indicated that this was due to small insertions or deletions. However, because such frameshifts lead to incorrect dS estimates, these genes were excluded from downstream analyses. 

In contrast, cases where the start or stop codon check returned FALSE were not excluded. Such differences may simply reflect mutations, and are expected in the context of Y chromosome degeneration. Therefore, these genes were retained for downstream analyses.

The genes removed for each species are listed below.

**S. latifolia**
- chr12_001860.1.fasta (divisible by 3: FALSE; start codon: TRUE; stop codon: TRUE)  
- chr12_001912.1.fasta (divisible by 3: FALSE; start codon: TRUE; stop codon: TRUE)  

**S. dioica**
- chr12_001860.1.fasta (divisible by 3: FALSE; start codon: TRUE; stop codon: TRUE)  

**S. diclinis**
- chr12_001860.1.fasta (divisible by 3: FALSE; start codon: TRUE; stop codon: TRUE)  
- chr12_001912.1.fasta (divisible by 3: FALSE; start codon: TRUE; stop codon: TRUE)  

**S. marizii**
- chr12_001860.1.fasta (divisible by 3: FALSE; start codon: TRUE; stop codon: TRUE)  

**S. heuffelii**
- chr12_001860.1.fasta (divisible by 3: FALSE; start codon: TRUE; stop codon: TRUE)  
- chr12_001912.1.fasta (divisible by 3: FALSE; start codon: FALSE; stop codon: TRUE)  


#### *Artemia* : sequence checks 

For *Artemia*, UTR trimming was not required, as explained above, because only exon sequences were extracted from the annotation.

However, sequences were still split by gene and checked for integrity before dS computation. 


In [ ]:
cd /PATH/Alignments/focal_species/perl_sequence_extraction/

multiple_fasta_file_ZW.sh ZW_sequences.fasta
wait
check_cutted_fasta.sh fasta_to_cut/
wait
grep 'FALSE' cut_check #see if you have problems 
grep 'FALSE' cut_check | wc -l #count the number of problems

All genes had lengths divisible by three, so no genes were excluded from downstream analyses.

### I.5.b dS computing

Pairwise dS values were computed using a custom script, `PAML_sex_chromosomes.sh`. This script formats X/Y (or Z/W) sequence pairs obtained from SEX-Detector and runs PAML to estimate synonymous divergence (dS). Internal stop codons are replaced by `NNN` prior to that estimation. The output file reports dS values as well as the number and positions of internal stop codons detected in the X (or Z) and Y (or W) sequences.

As input, the script requires one FASTA file per gene, containing first the X (or Z) sequence and then the Y (or W) sequence. These files were generated in the previous section.

In [ ]:
# Move to the directory containing gene-specific FASTA files
cd cutted_fasta/ # For Silene
cd fasta_to_cut/ # For Artemia

# Simplify FASTA headers by keeping only the gene name (remove everything after the first space)
sed -i 's/ .*$//' *.fasta 

# Run the PAML wrapper script to compute pairwise dS values
PAML_sex_chromosomes.sh 

# Inspect the output file (dS.txt)

# Identify genes with at least one internal stop codon in the X (or Z) sequence
grep ',.*,' dS.txt
grep ',.*,' dS.txt | wc -l

# Retain only genes without internal stop codons in the X (or Z) sequence for downstream analyses
grep -v ',.*,' dS.txt > gene_to_plot.txt 
wc -l  gene_to_plot.txt 


Genes whose X (or Z) sequence contained at least one internal stop codon were excluded from downstream analyses, as such sequences are likely pseudogenized and can lead to too elevated dS estimates.


## I.6 Topologies

### I.6.a create the corresponding outgroup fasta files

We first generated a list of genes shared by at least two species within each genus (*Silene* or *Artemia*). These shared gene lists were then used to extract the corresponding sequences from the outgroup species (*A. franciscana* and *S. viscosa*).

For the outgroups, several individuals were available. In addition, *A. franciscana* is a gonochoric species (separate males and females). Therefore, we generated one consensus sequence per gene for each outgroup. In this consensus, polymorphic positions were replaced by "N", so that only fixed sites were retained.

We also generated gene lists for each pair of focal species (pairwise comparisons). These lists will later be used to compute gene tree topologies.

#### 1. Identify shared genes

For each focal species, we first generated a list of XY (or ZW) genes. We then identified genes shared by at least two species within each genus. Finally, we created pairwise gene lists for all species comparisons.


In [ ]:
# For each focal species, generate a list of XY (or ZW) genes

cd /PATH/Alignments/focal_species/perl_sequence_extraction_filtered/fasta_to_cut/cutted_fasta # For Silene
cd /PATH/Alignments/focal_species/perl_sequence_extraction/fasta_to_cut # For Artemia 

ls *.fasta | sed 's/\.fasta$//' > /PATH/Alignments/list_gene_${Species}.txt

# Identify genes shared by at least two species within each genus

#for Artemia (two species only)
cd /PATH/Alignments/
awk '
    FNR==1 {file++}
    {count[$0]++}
    END {
        for (line in count)
            if (count[line] >= 2)
                print line
    }
' list_gene_artemia_sinica.txt list_gene_artemia_urmiana.txt | sort > gene_shared_artemia.txt

#for Silene (five species)
awk '                  
    FNR==1 {file++}
    {count[$0]++}
    END {
        for (line in count)
            if (count[line] >= 2)
                print line
    }
' list_gene_silene_latifolia.txt list_gene_silene_dioica.txt list_gene_silene_diclinis.txt list_gene_silene_heuffelii.txt list_gene_silene_marizii.txt| sort > gene_shared_silene_filtered.txt

# Generate gene lists shared between each pair of species (used later for pairwise topology analyses)

comm -12 <(sort list_gene_artemia_sinica.txt) <(sort list_gene_artemia_urmiana.txt) > gene_commun_artemia_sin_urm.txt 
# This file is identical to gene_shared_artemia.txt since only two Artemia species are analyzed

comm -12 <(sort list_gene_silene_latifolia.txt) <(sort list_gene_silene_dioica.txt) > gene_commun_silene_lat_dio_filtered.txt
comm -12 <(sort list_gene_silene_latifolia.txt) <(sort list_gene_silene_diclinis.txt) > gene_commun_silene_lat_dic_filtered.txt
comm -12 <(sort list_gene_silene_latifolia.txt) <(sort list_gene_silene_heuffelii.txt) > gene_commun_silene_lat_heu_filtered.txt
comm -12 <(sort list_gene_silene_latifolia.txt) <(sort list_gene_silene_marizii.txt) > gene_commun_silene_lat_mar_filtered.txt
comm -12 <(sort list_gene_silene_dioica.txt) <(sort list_gene_silene_diclinis.txt) > gene_commun_silene_dio_dic_filtered.txt
comm -12 <(sort list_gene_silene_dioica.txt) <(sort list_gene_silene_heuffelii.txt) > gene_commun_silene_dio_heu_filtered.txt
comm -12 <(sort list_gene_silene_dioica.txt) <(sort list_gene_silene_marizii.txt) > gene_commun_silene_dio_mar_filtered.txt
comm -12 <(sort list_gene_silene_diclinis.txt) <(sort list_gene_silene_heuffelii.txt) > gene_commun_silene_dic_heu_filtered.txt
comm -12 <(sort list_gene_silene_diclinis.txt) <(sort list_gene_silene_marizii.txt) > gene_commun_silene_dic_mar_filtered.txt
comm -12 <(sort list_gene_silene_heuffelii.txt) <(sort list_gene_silene_marizii.txt) > gene_commun_silene_heu_mar_filtered.txt

#### 2. Extract corresponding outgroup sequences

Using the shared gene list, we extracted the corresponding sequences from the outgroup FASTA file.


In [ ]:
# Extract fasta sequences for the outgroup species

genelist=/PATH/Alignments/gene_shared_genus.txt # List of shared genes (gene_shared_artemia.txt or gene_shared_silene_filtered.txt)
fasfile=outgroup.fas # Fasta file for the outgroup, output from read2snp (Artemia_franciscana.fas or Silene_viscosa.fas)
cd /PATH/Alignments/outgroup_species # Directory containing the outgroup fasta (Artemia_franciscana or Silene_viscosa)

# Create a directory to store fasta files for the shared genes
mkdir fastafile_shared

# For each gene in the shared gene list, extract its sequence from the outgroup fasta
for f in $(sed s'/.fasta//' $genelist); 
do 
	# Convert fasta to single-line sequences and extract the current gene
	awk '/^>/ {printf("\n%s\n", $0)} !/^>/ {printf("%s", $0)} END {printf("\n")}' $fasfile | grep -A1 $f > fastafile_shared/$f.fasta; 

done

# Some genes may be missing in the outgroup; remove any empty fasta files
ls fastafile_shared | wc -l
find fastafile_shared -type f -empty -delete
ls fastafile_shared | wc -l


#### 3. Build consensus sequences for the outgroup

Because multiple individuals were available for the outgroups, we generated one consensus sequence per gene. At each position, if all individuals shared the same nucleotide, it was retained. If polymorphism was detected, the position was replaced by "N".


In [ ]:
# Generate one consensus sequence per gene for the outgroup, replacing polymorphic sites with 'N'
# This ensures a single sequence per gene while retaining SNP positions for the focal species
cd fastafile_shared
for f in *.fasta; do

    # Extract the gene name from the first header line
    gene=$(grep '^>' "$f" | head -n1 | cut -d'|' -f1 | sed 's/^>//')
    echo ">$gene" > "${f%.fasta}_consensus.fasta"

    # Build the consensus sequence
    awk '
    /^>/ {next}                             # Skip fasta headers
    { seq[++n] = $0; len = length($0) }     # Store sequence lines in array, n counts sequences

    END {
        for (i = 1; i <= len; i++) {
            delete letters
            # Count the nucleotides at each position across all sequences
            for (j = 1; j <= n; j++) {
                ch = substr(seq[j], i, 1)
                letters[ch]++
            }

            # Count the number of unique nucleotides at this position
            nkeys = 0
            for (x in letters) nkeys++

            # Define consensus rules
            if (nkeys == 1) {
                # All nucleotides identical → keep that nucleotide
                for (x in letters) printf "%s", x
            } else if (nkeys == 2 && letters["N"] > 0) {
                # Two nucleotides, one is 'N' → keep the non-N nucleotide
                for (x in letters) if (x != "N") printf "%s", x
            } else {
                # More than 2 nucleotides, or 2 nucleotides without 'N' → set as 'N'
                printf "N"
            }
        }
        printf "\n"
    }' "$f" >> "${f%.fasta}_consensus.fasta"
done

# Remove original fasta files and rename consensus files to standard fasta names
find . -maxdepth 1 -type f -name "*.fasta" ! -name "*_consensus.fasta" -delete
for f in *_consensus.fasta; do
  mv "$f" "${f/_consensus.fasta/.fasta}"
done

# Convert consensus fasta sequences into CSV format for easier downstream processing
mkdir -p fastafile_final
for f in *.fasta; do
  output="fastafile_final/${f%.fasta}.csv"
  awk '
    /^>/ {next}  # Skip fasta headers
    {
      for (i = 1; i <= length($0); i++) {
        print substr($0, i, 1)  # Print each nucleotide on a separate line
      }
    }
  ' "$f" > "$output"
done


#### 4. Update gene lists after filtering

Genes without a corresponding outgroup sequence were removed from the shared gene lists. Corrected gene lists were generated for downstream analyses.


In [ ]:
# Filter gene lists by removing genes that are missing in the outgroup sequences
# Create a filtered list of genes actually present in the outgroup
ls fastafile_shared | grep '\.fasta$' | sed 's/\.fasta$//' >  /PATH/Alignments/gene_shared_genus_corrected.txt #(gene_shared_artemia_corrected.txt or gene_shared_silene_filtered_corrected.txt)

# For Artemia: keep only genes shared with the outgroup
grep -Fxf <(ls fastafile_shared/ | sed 's/\.fasta$//') \
  /PATH/Alignments/gene_commun_artemia_urm.txt \
  > /PATH/Alignments/gene_commun_artemia_sin_urm_corrected.txt

# For Silene: apply the same filtering for all pairwise gene lists
for f in /PATH/Alignments/gene_commun_*.txt; do
    outfile="${f%.txt}_corrected.txt"
    grep -Fxf <(ls /PATH/Alignments/Silene_viscosa/fastafile_shared | sed 's/\.fasta$//') \
        "$f" > "$outfile"
done

### I.6.b Compute topologies

For the final step of the pipeline, we compute gene- and SNP-level topologies between species pairs. This uses two custom scripts:

- **`topo_creator.R`**: Creates two tables per species pair:
  - Gene-level summary, reporting the count of each topology per gene.
  - SNP-level detailed table, listing the topology observed at each SNP.

- **`SNP_duplicate.R`**: Identifies SNPs likely to be artifacts due to duplication or genotyping inconsistencies. For example, if both parents are CT but all children are also CT instead of segregating as 1/4 CC, 1/4 TT, 1/2 CT, these SNPs are flagged.


In [ ]:
# Compute topologies for a species pair
# topo_creator.R requires the following arguments:
# 1) Directory containing sequences for species 1
# 2) Directory containing sequences for species 2
# 3) Name of the species pair (used as a prefix for output files)
# 4) List of shared genes for the species pair
# 5) Output directory for topology tables
# 6) Directory containing outgroup sequences in CSV format

# Example for Artemia:
topo_creator.R /PATH/Alignments/Artemia_sinica/perl_sequence_extraction/fasta_to_cut/ /PATH/Alignments/Artemia_urmiana/perl_sequence_extraction/fasta_to_cut/ "topo_sin_urm" /PATH/Alignments/gene_commun_artemia_sin_urm_corrected.txt /PATH/Topologies/ /PATH/Alignments/Artemia_franciscana/fastafile_shared/fastafile_final

# Example for Silene (latifolia-dioica pair; repeat for all species pairs):
topo_creator.R /PATH/Alignments/Silene_latifolia/perl_sequence_extraction_filtered/fasta_to_cut/cutted_fasta/ /PATH/Alignments/Silene_dioica/perl_sequence_extraction_filtered/fasta_to_cut/cutted_fasta/ "topo_lat_dio_filtered" /PATH/Alignments/gene_commun_silene_lat_dio_filtered_corrected.txt /PATH/Topologies/ /PATH/Alignments/Silene_viscosa/fastafile_shared/fastafile_final

# Run SNP_duplicate.R for each species to identify potentially duplicated or inconsistent SNPs
# Arguments:
# 1) File containing detailed SNP information for the family
# 2) List of shared genes to consider (e.g., gene_shared_artemia.txt or gene_shared_silene_filtered.txt)
SNP_duplicate.R /PATH/Alignments/focal_species/${FamilyFile}_perl_SNPs_detail.txt /PATH/Alignments/gene_shared_genus.txt


**Notes:** You will notice that all files created for *Silene* since the sequence extraction step carry the suffix `filtered`, as these correspond to sequences obtained after population data cleaning. This suffix was kept for clarity, even if it makes the filenames slightly longer. The same steps were also performed on the unfiltered sequences (without the `filtered` suffix) for comparison, and these unfiltered files were used for a few unpublished analyses (mainly to compare results with and without population data cleaning).

## I.7 File organisation 

Now we have all the necessary files to begin analysis. To make things easier, the files were reorganized as follows. A main directory `Analyses` was created, containing subdirectories with all files needed for the analyses. Some names were slightly shortened for readability, but it remains clear what each file represents (mainly supress the genus name)  


<pre>
Analyses
├── Snp_detail
│   ├── Artemia_sinica_Family_perl_SNPs_detail.txt
│   ├── Artemia_urmiana_Family_perl_SNPs_detail.txt
│   ├── Silene_diclinis_Family_perl_SNPs_detail.txt
│   ├── Silene_dioica_Family_perl_SNPs_detail.txt
│   ├── Silene_heuffelli_Family_perl_SNPs_detail.txt
│   ├── Silene_latifolia_Family_perl_SNPs_detail.txt
│   └── Silene_marizii_Family_perl_SNPs_detail.txt
├── assignment
│   ├── Artemia_sinica_Family_perl_assignment.txt
│   ├── Artemia_urmiana_Family_perl_assignment.txt
│   ├── Silene_diclinis_Family_perl_assignment.txt
│   ├── Silene_dioica_Family_perl_assignment.txt
│   ├── Silene_heuffelli_Family_perl_assignment.txt
│   ├── Silene_latifolia_Family_perl_assignment.txt
│   └── Silene_marizii_Family_perl_assignment.txt
├── dS_gene_to_plot
│   ├── gene_to_plot_diclinis.txt
│   ├── gene_to_plot_diclinis_filtered.txt
│   ├── gene_to_plot_dioica.txt
│   ├── gene_to_plot_dioica_filtered.txt
│   ├── gene_to_plot_heuffelii.txt
│   ├── gene_to_plot_heuffelii_filtered.txt
│   ├── gene_to_plot_latifolia.txt
│   ├── gene_to_plot_latifolia_filtered.txt
│   ├── gene_to_plot_marizii.txt
│   ├── gene_to_plot_marizii_filtered.txt
│   ├── gene_to_plot_sinica.txt
│   └── gene_to_plot_urmiana.txt
├── gtf_file
│   ├── S.latifolia_v4.0.gff.gff3_polished_no_Y
│   ├── artemia_sinica_annotation_2024.filtered.gtf
├── list_gene
│   ├── gene_commun_dic_heu.txt
│   ├── gene_commun_dic_heu_corrected.txt
│   ├── gene_commun_dic_heu_filtered.txt
│   ├── gene_commun_dic_heu_filtered_corrected.txt
│   ├── gene_commun_dic_mar.txt
│   ├── gene_commun_dic_mar_corrected.txt
│   ├── gene_commun_dic_mar_filtered.txt
│   ├── gene_commun_dic_mar_filtered_corrected.txt
│   ├── gene_commun_dio_dic.txt
│   ├── gene_commun_dio_dic_corrected.txt
│   ├── gene_commun_dio_dic_filtered.txt
│   ├── gene_commun_dio_dic_filtered_corrected.txt
│   ├── gene_commun_dio_heu.txt
│   ├── gene_commun_dio_heu_corrected.txt
│   ├── gene_commun_dio_heu_filtered.txt
│   ├── gene_commun_dio_heu_filtered_corrected.txt
│   ├── gene_commun_dio_mar.txt
│   ├── gene_commun_dio_mar_corrected.txt
│   ├── gene_commun_dio_mar_filtered.txt
│   ├── gene_commun_dio_mar_filtered_corrected.txt
│   ├── gene_commun_heu_mar.txt
│   ├── gene_commun_heu_mar_corrected.txt
│   ├── gene_commun_heu_mar_filtered.txt
│   ├── gene_commun_heu_mar_filtered_corrected.txt
│   ├── gene_commun_lat_dic.txt
│   ├── gene_commun_lat_dic_corrected.txt
│   ├── gene_commun_lat_dic_filtered.txt
│   ├── gene_commun_lat_dic_filtered_corrected.txt
│   ├── gene_commun_lat_dio.txt
│   ├── gene_commun_lat_dio_corrected.txt
│   ├── gene_commun_lat_dio_filtered.txt
│   ├── gene_commun_lat_dio_filtered_corrected.txt
│   ├── gene_commun_lat_heu.txt
│   ├── gene_commun_lat_heu_corrected.txt
│   ├── gene_commun_lat_heu_filtered.txt
│   ├── gene_commun_lat_heu_filtered_corrected.txt
│   ├── gene_commun_lat_mar.txt
│   ├── gene_commun_lat_mar_corrected.txt
│   ├── gene_commun_lat_mar_filtered.txt
│   ├── gene_commun_lat_mar_filtered_corrected.txt
│   ├── gene_commun_sin_urm.txt
│   ├── gene_commun_sin_urm_corrected.txt
│   ├── list_gene_XY_perl_diclinis.csv
│   ├── list_gene_XY_perl_dioica.csv
│   ├── list_gene_XY_perl_heuffelii.csv
│   ├── list_gene_XY_perl_latifolia.csv
│   ├── list_gene_XY_perl_marizii.csv
│   ├── list_gene_ZW_perl_sinica.csv
│   ├── list_gene_ZW_perl_urmiana.csv
│   ├── list_gene_dic.txt
│   ├── list_gene_dic_filtered.txt
│   ├── list_gene_dio.txt
│   ├── list_gene_dio_filtered.txt
│   ├── list_gene_heu.txt
│   ├── list_gene_heu_filtered.txt
│   ├── list_gene_lat.txt
│   ├── list_gene_lat_filtered.txt
│   ├── list_gene_mar.txt
│   ├── list_gene_mar_filtered.txt
├── sex_linked_details
│   ├── Artemia_sinica_Family_perl_sex-linked_detail.txt
│   ├── Artemia_urmiana_Family_perl_sex-linked_detail.txt
│   ├── Silene_diclinis_Family_perl_sex-linked_detail.txt
│   ├── Silene_dioica_Family_perl_sex-linked_detail.txt
│   ├── Silene_heuffelli_Family_perl_sex-linked_detail.txt
│   ├── Silene_latifolia_Family_perl_sex-linked_detail.txt
│   └── Silene_marizii_Family_perl_sex-linked_detail.txt
├── snp_duplicate
│   ├── prop_snp_duplicate_diclinis.csv
│   ├── prop_snp_duplicate_dioica.csv
│   ├── prop_snp_duplicate_heuffelii.csv
│   ├── prop_snp_duplicate_latifolia.csv
│   ├── prop_snp_duplicate_marizii.csv
│   ├── prop_snp_duplicate_sinica.csv
│   └── prop_snp_duplicate_urmiana.csv
├── snp_to_remove
│   ├── snp_to_remove_diclinis.csv
│   ├── snp_to_remove_dioica.csv
│   ├── snp_to_remove_heuffelii.csv
│   ├── snp_to_remove_latifolia.csv
│   └── snp_to_remove_marizii.csv
└── topo
    ├── topo_dic_heu.csv
    ├── topo_dic_heu_filtered.csv
    ├── topo_dic_mar.csv
    ├── topo_dic_mar_filtered.csv
    ├── topo_dio_dic.csv
    ├── topo_dio_dic_filtered.csv
    ├── topo_dio_heu.csv
    ├── topo_dio_heu_filtered.csv
    ├── topo_dio_mar.csv
    ├── topo_dio_mar_filtered.csv
    ├── topo_heu_mar.csv
    ├── topo_heu_mar_filtered.csv
    ├── topo_lat_dic.csv
    ├── topo_lat_dic_filtered.csv
    ├── topo_lat_dio.csv
    ├── topo_lat_dio_filtered.csv
    ├── topo_lat_heu.csv
    ├── topo_lat_heu_filtered.csv
    ├── topo_lat_mar.csv
    ├── topo_lat_mar_filtered.csv
    └── topo_sin_urm.csv
    <pre>

# II. Analyses 

We will mainly use R version 4.5.2 (2025-10-31) to do our analysis, with the following packages. 

In [ ]:
#%use R
# %output: False

library(patchwork)
library(reshape2)
library(zoo)
library(tidyverse)

## II.1 Preliminary steps for all analyses

### II.1.a Settings

In [ ]:
getwd()

In [ ]:
#%use R
setwd("/PATH") # Set the working directory (For us the directory Analyses as explained in section I.7)
set.seed(123)


### II.1.b Treatment of gtf and gff file 

In [ ]:
#%use R

file_gtf_artemia <- "gtf_file/artemia_sinica_annotation_2024.filtered.gtf"
file_gff_silene <- "gtf_file/S.latifolia_v4.0.gff.gff3_polished_no_Y"


gtf_function <- function(file_gtf) {
  
  data_gtf <- read.table(file_gtf, sep="\t", header=FALSE, comment.char="#", stringsAsFactors=FALSE)
  colnames(data_gtf) <- c("Chromosome", "Source", "Type", "Start", "End", "Score", "Strand", "Phase", "Attributes")
  data_gtf <- data_gtf %>% separate(col = Attributes, 
           into = c("gene_ID", "transcript_ID", "ID", "Parent"),
           sep = ";",
           convert = TRUE, 
           extra = "drop") 
  data_gtf$ID <- gsub("ID ", "", data_gtf$ID)
  data_gtf$gene_ID <- gsub("gene_id ", "", data_gtf$gene_ID)
  data_gtf$transcript_ID <- gsub("transcript_id ", "", data_gtf$transcript_ID)
  data_gtf$Parent <- gsub("Parent ", "", data_gtf$Parent)
  data_gtf <- data_gtf %>%
  mutate(ID = str_trim(ID)) %>% 
  filter(startsWith(Chromosome,"Chromosome")) #remove scaffolds and contigs that are not assigned to a chromosome
  return(data_gtf)
  }



gff_function<-function(file_gff){
  data_gff <- read.table(file_gff, sep="\t", header=FALSE, comment.char="#", stringsAsFactors=FALSE)
  colnames(data_gff) <- c("Chromosome", "Source", "Type", "Start", "End", "Score", "Strand", "Phase", "Attributes")
  data_gff <- data_gff %>%
  separate(col = Attributes, 
           into = c("ID", "Coverage", "Sequence_ID", "Valid_ORFs", "Extra_copy_number", "Copy_num_ID"),
           sep = ";",
           convert = TRUE, 
           extra = "drop") 
  data_gff$ID <- gsub("^ID=_", "", data_gff$ID)
  return(data_gff)
}

data_gtf_artemia<-gtf_function(file_gtf_artemia)
data_gff_silene<-gff_function(file_gff_silene)

### II.1.c Usefull things

In [ ]:
species=c("Artemia_sinica","Artemia_urmiana", "Silene_latifolia", "Silene_dioica", "Silene_marizii", "Silene_heuffelli", "Silene_diclinis")
species_short=c("A.sinica","A.urmiana", "S.latifolia", "S.dioica", "S.marizii", "S.heuffelli", "S.diclinis")
names=c("sin","urm","lat","dio","dic","heu","mar")
pairs=c("sin_urm", "lat_dio", "lat_dic", "lat_heu", "lat_mar",
  "dio_dic", "dio_heu", "dio_mar", "dic_heu", "dic_mar", "heu_mar")

pairs_name=c("A.sinica-A.urmiana", "S.latifolia-S.dioica", "S.latifolia-S.diclinis", "S.latifolia-S.heuffelii", "S.latifolia-S.marizii",
  "S.dioica-S.diclinis", "S.dioica-S.heuffelii", "S.dioica-S.marizii", "S.diclinis-S.heuffelii", "S.diclinis-S.marizii", "S.heuffelii-S.marizii")

# limits found in the litterature for silene
arrows <- data.frame(x = c(5769054, 10069580, 15830480, 36776284, 315567532), y = c(0,0,0,0,0), yend = c(0.1,0.1,0.1,0.1,0.1))
# limits found in the litterature for artemia 
arrows_artemia <- data.frame(
  x = c(35322500, 63525001, 88115001,100585001),
  y = c(0, 0, 0, 0),
  yend = c(0.1, 0.1, 0.1,0.1)
)


## II.2 Which genes are sex-linked ? 

### II.2.a Data importation

In [ ]:
for (sp in species) {
  file_path <- paste0("assignment/", sp, "_Family_perl_assignment.txt")
  name <- paste0("data_", sub(".*_", "", sp))
  
  assign(name,read.table(file_path,header = TRUE,sep = "\t",stringsAsFactors = FALSE,comment.char = ""))
}

### II.2.b Data treatment

In [ ]:
levels_silene=c("chr1","chr2","chr3","chr4","chr5","chr6","chr7","chr8","chr9","chr10","chr11","chrX")
levels_artemia=c("Chromosome_1","Chromosome_2","Chromosome_3","Chromosome_4","Chromosome_5","Chromosome_6","Chromosome_7","Chromosome_8","Chromosome_9","Chromosome_10","Chromosome_11","Chromosome_12","Chromosome_13","Chromosome_14","Chromosome_15","Chromosome_16","Chromosome_17","Chromosome_18","Chromosome_19","Chromosome_20","Chromosome_21")

data_function<- function(data,species,data_gff,levels) {
  data$contig <- gsub("^_", "", data$contig)
  data <- data %>%
    left_join(data_gff, by = c("contig" = "ID")) %>%
    select(Chromosome, contig, assignment, Start, End, probability_autosomal, probability_sex.linked, probability_hemizygous) %>%
    mutate(Length = End - Start) %>%
    mutate(middle = Start + (End - Start) / 2) %>%
    melt(id.vars = c("Chromosome", "contig", "assignment", "Start", "End", "Length", "middle")) %>%
    filter(assignment != "lack-information") %>%
    mutate(species=species) %>%
    select(Chromosome,assignment,Start,variable,value,species)
  
  data$assignment <- as.factor(data$assignment)
  data$Chromosome <- factor(data$Chromosome, levels= levels)
  data$species <- as.factor(data$species)
  data$variable <- as.factor(data$variable)

  return(data)
}

data_latifolia<-data_function(data_latifolia,"S.latifolia",data_gff_silene,levels_silene)
data_dioica<-data_function(data_dioica,"S.dioica",data_gff_silene,levels_silene)
data_marizii<-data_function(data_marizii,"S.marizii",data_gff_silene,levels_silene)
data_heuffelli<-data_function(data_heuffelli,"S.heuffelli",data_gff_silene,levels_silene)
data_diclinis<-data_function(data_diclinis,"S.diclinis",data_gff_silene,levels_silene)
data_sinica<-data_function(data_sinica,"A.sinica",data_gtf_artemia,levels_artemia)
data_urmiana<-data_function(data_urmiana,"A.urmiana",data_gtf_artemia,levels_artemia)


### II.2.c plots

In [ ]:
data_proba_xy<-rbind(data_latifolia,data_dioica,data_heuffelli,data_marizii,data_diclinis) 
data_proba_zw<-rbind(data_sinica,data_urmiana)  

P1 <- ggplot(mutate(filter(donnees_proba_xy,str_starts(Chromosome, "chr")), Start = Start/1e7), aes(x = Start, y = value)) + #filter(donnees_proba_xy, Chromosome=="chrX")
    geom_smooth(aes(color = variable), size = 1) +
    xlab("Position (bp x 10^7)") +
    ylab("Probability") +
    ylim(0, 1) + 
    facet_grid(species~ Chromosome, scales = "free_x") +
    theme_bw() +
    theme(legend.position = "bottom")

P2 <- ggplot(mutate(donnees_proba_zw, Start = Start/1e7), aes(x = Start, y = value)) +  #filter(donnees_proba_xy, Chromosome=="chrX")
    geom_smooth(aes(color = variable), size = 1) +
    xlab("Position (bp x 10^7)") +
    ylab("Probability") +
    ylim(0, 1) + 
    facet_grid(species~ Chromosome, scales = "free_x") +
    theme_bw()+
    theme(legend.position = "bottom")

P1
P2

## II.3 Silene population cleaning

### II.3.a Data importation 

In [ ]:
snp_to_remove_dic <- read_csv("snp_to_remove/snp_to_remove_diclinis.csv",  col_names = TRUE) 
snp_to_remove_lat <- read_csv("snp_to_remove/snp_to_remove_latifolia.csv",  col_names = TRUE) 
snp_to_remove_dio <- read_csv("snp_to_remove/snp_to_remove_dioica.csv",  col_names = TRUE) 
snp_to_remove_heu <- read_csv("snp_to_remove/snp_to_remove_heuffelii.csv",  col_names = TRUE) 
snp_to_remove_mar <- read_csv("snp_to_remove/snp_to_remove_marizii.csv",  col_names = TRUE) 

sex_linked_detail_lat <- read_delim("sex_linked_details/Silene_latifolia_Family_perl_sex-linked_detail.txt",  delim = "\t")
sex_linked_detail_dic <- read_delim("sex_linked_details/Silene_diclinis_Family_perl_sex-linked_detail.txt",  delim = "\t")
sex_linked_detail_dio <- read_delim("sex_linked_details/Silene_dioica_Family_perl_sex-linked_detail.txt",  delim = "\t")
sex_linked_detail_heu <- read_delim("sex_linked_details/Silene_heuffelli_Family_perl_sex-linked_detail.txt",  delim = "\t")
sex_linked_detail_mar <- read_delim("sex_linked_details/Silene_marizii_Family_perl_sex-linked_detail.txt",  delim = "\t")

### II.3.b Data treatment

Inferred Y snp that are found in population females (false Y) 

In [ ]:
percent_plot_function <- function(snp_to_remove, sex_linked_detail) {
  snp_count<-snp_counter_function(sex_linked_detail)
  snp_to_remove <- snp_to_remove %>% group_by(contig_name) %>% 
    filter(Yfemcount>0) %>% 
    summarise(fake_snp_count = n(), .groups = "drop")%>%
    left_join(snp_count, by = c("contig_name"="contig_name")) 
  snp_to_remove$contig_name <- gsub("^_", "", snp_to_remove$contig_name)
  snp_to_remove <- snp_to_remove%>%
    left_join(data_gff_silene,by = c("contig_name" = "ID")) %>%
    select(Chromosome, contig_name, Start, End, fake_snp_count,snp_count) %>%
    mutate(percent_snp=fake_snp_count/snp_count) %>%
    mutate(Length=End-Start+1) %>%
    filter(Chromosome=="chrX" )

  return(snp_to_remove)}

perc_snp_lat <- percent_plot_function(snp_to_remove_lat,sex_linked_detail_lat) %>% mutate(Species="S.latifolia")
perc_snp_dic <- percent_plot_function(snp_to_remove_dic,sex_linked_detail_dic) %>% mutate(Species="S.diclinis")
perc_snp_dio <- percent_plot_function(snp_to_remove_dio,sex_linked_detail_dio) %>% mutate(Species="S.dioica")
perc_snp_heu <- percent_plot_function(snp_to_remove_heu,sex_linked_detail_heu) %>% mutate(Species="S.heuffelii")
perc_snp_mar <- percent_plot_function(snp_to_remove_mar,sex_linked_detail_mar) %>% mutate(Species="S.marizii")

perc_snp<-rbind(perc_snp_lat,perc_snp_dio,perc_snp_dic,perc_snp_heu,perc_snp_mar)

Inferred Y snp that are missing in population males (missing Y). We didn't remove them but just wanted to check.

In [ ]:
percent_plot_function_Y <- function(snp_to_remove, sex_linked_detail) {
  snp_count<-snp_counter_function(sex_linked_detail)
  snp_to_remove <- snp_to_remove %>% group_by(contig_name) %>% 
    filter(noYmalcount>0) %>%
    summarise(fake_snp_count = n(), .groups = "drop")%>%
    left_join(snp_count, by = c("contig_name"="contig_name"))
  snp_to_remove$contig_name <- gsub("^_", "", snp_to_remove$contig_name)
  snp_to_remove <- snp_to_remove%>%
    left_join(data_gff_silene,by = c("contig_name" = "ID")) %>%
    select(Chromosome, contig_name, Start, End, fake_snp_count,snp_count) %>%
    mutate(percent_snp=fake_snp_count/snp_count) %>%
    mutate(Length=End-Start+1) %>%
    filter(Chromosome=="chrX" )

  return(snp_to_remove)}

perc_snp_Y_lat <- percent_plot_function_Y(snp_to_remove_lat,sex_linked_detail_lat) %>% mutate(Species="S.latifolia")
perc_snp_Y_dic <- percent_plot_function_Y(snp_to_remove_dic,sex_linked_detail_dic) %>% mutate(Species="S.diclinis")
perc_snp_Y_dio <- percent_plot_function_Y(snp_to_remove_dio,sex_linked_detail_dio) %>% mutate(Species="S.dioica")
perc_snp_Y_heu <- percent_plot_function_Y(snp_to_remove_heu,sex_linked_detail_heu) %>% mutate(Species="S.heuffelii")
perc_snp_Y_mar <- percent_plot_function_Y(snp_to_remove_mar,sex_linked_detail_mar) %>% mutate(Species="S.marizii")

perc_snp_Y<-rbind(perc_snp_Y_lat,perc_snp_Y_dio,perc_snp_Y_dic,perc_snp_Y_heu,perc_snp_Y_mar)

### II.3.c Plots

In [ ]:
R1 <- ggplot() + 
  geom_point(data=perc_snp,aes(x=Start,y=percent_snp) ,size = log(perc_snp$snp_count),alpha = log(perc_snp$snp_count)/5, shape=16) +
    xlab("position (pdb)") +
    ylab("probabilité") +
    ylim(0, 1) + 
    xlim(0, 350000000) + 
    ggtitle("False Y") +
    facet_grid(rows= vars(Species), scales = "free_x") +
    theme_bw()

R2 <- ggplot() + 
    geom_point(data=perc_snp_Y,aes(x=Start,y=percent_snp) ,size = log(perc_snp_Y$snp_count),alpha = log(perc_snp_Y$snp_count)/5, shape=16) +
    xlab("position (pdb)") +
    ylab("probabilité") +
    ylim(0, 1) + 
    xlim(0, 350000000) + 
    ggtitle("Missing Y Y") +
    facet_grid(rows= vars(Species), scales = "free_x") +
    theme_bw()

R1
R2

## II.4 Working on dS

### II.4.a Data importation 

#### 1. Sex-linked detail files

In [ ]:
sex_linked_detail_sin<- "sex_linked_details/Artemia_sinica_Family_perl_sex-linked_detail.txt"
sex_linked_detail_urm<-"sex_linked_details/Artemia_urmiana_Family_perl_sex-linked_detail.txt"
sex_linked_detail_lat<- "sex_linked_details/Silene_latifolia_Family_perl_sex-linked_detail.txt"
sex_linked_detail_dic<- "sex_linked_details/Silene_diclinis_Family_perl_sex-linked_detail.txt"
sex_linked_detail_dio<- "sex_linked_details/Silene_dioica_Family_perl_sex-linked_detail.txt"
sex_linked_detail_heu<- "sex_linked_details/Silene_heuffelli_Family_perl_sex-linked_detail.txt"
sex_linked_detail_mar<- "sex_linked_details/Silene_marizii_Family_perl_sex-linked_detail.txt"

gene_filter_sin <- read_delim(sex_linked_detail_sin,  delim = "\t")
gene_filter_urm <- read_delim(sex_linked_detail_urm,  delim = "\t")
gene_filter_lat <- read_delim(sex_linked_detail_lat,  delim = "\t")
gene_filter_dic <- read_delim(sex_linked_detail_dic,  delim = "\t")
gene_filter_dio <- read_delim(sex_linked_detail_dio,  delim = "\t")
gene_filter_heu <- read_delim(sex_linked_detail_heu,  delim = "\t")
gene_filter_mar <- read_delim(sex_linked_detail_mar,  delim = "\t")

#### 2. dS

In [ ]:
file_dS_sin <- "dS_gene_to_plot/gene_to_plot_sinica.txt"
file_dS_urm <- "dS_gene_to_plot/gene_to_plot_urmiana.txt"

file_dS_lat_no_filter <- "dS_gene_to_plot/gene_to_plot_latifolia.txt"
file_dS_dic_no_filter <- "dS_gene_to_plot/gene_to_plot_diclinis.txt"
file_dS_dio_no_filter <- "dS_gene_to_plot/gene_to_plot_dioica.txt"
file_dS_heu_no_filter <- "dS_gene_to_plot/gene_to_plot_heuffelii.txt"
file_dS_mar_no_filter <- "dS_gene_to_plot/gene_to_plot_marizii.txt"

file_dS_lat <- "dS_gene_to_plot/gene_to_plot_latifolia_filtered.txt"
file_dS_dic <- "dS_gene_to_plot/gene_to_plot_diclinis_filtered.txt"
file_dS_dio <- "dS_gene_to_plot/gene_to_plot_dioica_filtered.txt"
file_dS_heu <- "dS_gene_to_plot/gene_to_plot_heuffelii_filtered.txt"
file_dS_mar <- "dS_gene_to_plot/gene_to_plot_marizii_filtered.txt"

dS_sin <- read.table(file_dS_sin, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_urm <- read.table(file_dS_urm, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)

dS_lat <- read.table(file_dS_lat, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_dic <- read.table(file_dS_dic, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_dio <- read.table(file_dS_dio, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_heu <- read.table(file_dS_heu, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_mar <- read.table(file_dS_mar, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)

dS_lat_no_filter <- read.table(file_dS_lat_no_filter, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_dic_no_filter <- read.table(file_dS_dic_no_filter, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_dio_no_filter <- read.table(file_dS_dio_no_filter, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_heu_no_filter <- read.table(file_dS_heu_no_filter, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)
dS_mar_no_filter <- read.table(file_dS_mar_no_filter, sep="\t", header=TRUE, comment.char="#", stringsAsFactors=FALSE)

### II.4.b Data treatment 

#### 1. gtf and gff file simplification 

In [ ]:
data_gtf_artemia <- data_gtf_artemia %>% filter(Type =="transcript")
data_gff_silene  <- data_gff_silene %>% filter(Type =="mRNA")

#### 2. Computation of expression ratio 

We will only keep Y (or W) that are expressed at least at one-quarter the level of X (or Z).

In [ ]:
gene_filter_maker_artemia <- function(gene_filter_data) {
  gene_filter_data <- gene_filter_data %>% 
    filter(SNP_type %in% c("XY" , "XXY" , "XXXY"))%>%
    select(contig_name, position, contains("_female"), contains("mother")) %>%
    select(contig_name, position, contains("expr"))%>%
    select(- contains("total")) %>%
    mutate(across(contains("_sorted"), 
                  ~ ifelse(is.na(.) | . == "N" | get(str_replace(cur_column(), "_Z_expr$", "_W_expr")) == "N",
                           NA, 
                           get(str_replace(cur_column(), "_Z_expr$", "_W_expr")) / .),
                  .names = "{.col}_ratio")) %>%
    select(contig_name,position,contains("Z_expr_ratio")) %>%
    mutate(mean_ratio = rowMeans(select(., -contig_name, -position), na.rm = TRUE)) %>%
    group_by(contig_name) %>%
    summarize(mean=mean(mean_ratio))
  gene_filter_data$contig_name <- sub("_", "", gene_filter_data$contig_name) 
  return(gene_filter_data)
}

gene_filter_maker_silene <- function(gene_filter_data) {
  gene_filter_data <- gene_filter_data %>% 
    filter(SNP_type %in% c("XY" , "XXY" , "XXXY"))%>%
    select(contig_name, position, contains("_male"), contains("father")) %>%
    select(contig_name, position, contains("expr"))%>%
    select(- contains("total")) %>%
    mutate(across(contains("_sorted"), 
                  ~ ifelse(is.na(.) | . == "N" | get(str_replace(cur_column(), "_X_expr$", "_Y_expr")) == "N",
                           NA, 
                           get(str_replace(cur_column(), "_X_expr$", "_Y_expr")) / .),
                  .names = "{.col}_ratio")) %>%
    select(contig_name,position,contains("X_expr_ratio")) %>%
    mutate(mean_ratio = rowMeans(select(., -contig_name, -position), na.rm = TRUE)) %>%
    group_by(contig_name) %>%
    summarize(mean=mean(mean_ratio))
  gene_filter_data$contig_name <- sub("_", "", gene_filter_data$contig_name) 
  return(gene_filter_data)
}

gene_filter_sin <- gene_filter_maker_artemia(gene_filter_sin)
gene_filter_urm <- gene_filter_maker_artemia(gene_filter_urm)

gene_filter_lat <- gene_filter_maker_silene(gene_filter_lat)
gene_filter_dic <- gene_filter_maker_silene(gene_filter_dic)
gene_filter_dio <- gene_filter_maker_silene(gene_filter_dio)
gene_filter_heu <- gene_filter_maker_silene(gene_filter_heu)
gene_filter_mar <- gene_filter_maker_silene(gene_filter_mar)


#### 3. Final table for dS before plots. 

We filter base on the ratio of Z/W or X/Y expression and keep only the sex chromosomes

In [ ]:
dS_maker <- function(dS_data, data_gtf, gene_filter_data, species) {
  dS_data <- dS_data  %>%
    left_join(data_gtf, by = c("fasta_file_Name" = "ID")) %>% 
    left_join(gene_filter_data, by=c("fasta_file_Name" = "contig_name")) %>% 
    filter(mean >=0.25) %>% 
    select(Chromosome, fasta_file_Name,dS,Start,End) %>%
    mutate(Middle=Start + (End-Start)/2) %>%
    mutate(Length=End-Start+1) %>%
    mutate(GeneID = str_remove(fasta_file_Name, "\\.t\\d+$")) %>%
    group_by(GeneID) %>%
    filter(Length == max(Length)) %>% #to keep only the longest transcript
    slice_sample(n = 1) %>%  #if two transcripts have the same length, chose randomly
    ungroup() %>%
    mutate(Species=species)

  return(dS_data)
}

dS_maker_artemia <- function(dS_data, data_gtf, gene_filter_data, species) {
  dS_data<-dS_maker(dS_data, data_gtf, gene_filter_data, species)%>% filter(Chromosome=="Chromosome_1")
}

dS_maker_silene <- function(dS_data, data_gtf, gene_filter_data, species) {
  dS_data<-dS_maker(dS_data, data_gtf, gene_filter_data, species) %>% filter(startsWith(fasta_file_Name,"chr12")) %>% filter(!(fasta_file_Name%in%c("chr12_001954.1","chr12_001860.1","chr12_001912.1")))
}

dS_sin <- dS_maker_artemia(dS_sin, data_gtf_artemia, gene_filter_sin,"A.sinica")
dS_urm <- dS_maker_artemia(dS_urm, data_gtf_artemia, gene_filter_urm, "A.urmiana")

dS_lat <- dS_maker_silene(dS_lat, data_gff_silene, gene_filter_lat,"S.latifolia") 
dS_dic <- dS_maker_silene(dS_dic, data_gff_silene, gene_filter_dic,"S.diclinis")
dS_dio <- dS_maker_silene(dS_dio, data_gff_silene, gene_filter_dio,"S.dioica")
dS_heu <- dS_maker_silene(dS_heu, data_gff_silene, gene_filter_heu,"S.heuffelii")
dS_mar <- dS_maker_silene(dS_mar, data_gff_silene, gene_filter_mar,"S.marizii")

dS_lat_no_filter <- dS_maker_silene(dS_lat_no_filter, data_gff_silene, gene_filter_lat,"S.latifolia") 
dS_dic_no_filter <- dS_maker_silene(dS_dic_no_filter, data_gff_silene, gene_filter_dic,"S.diclinis")
dS_dio_no_filter <- dS_maker_silene(dS_dio_no_filter, data_gff_silene, gene_filter_dio,"S.dioica")
dS_heu_no_filter <- dS_maker_silene(dS_heu_no_filter, data_gff_silene, gene_filter_heu,"S.heuffelii")
dS_mar_no_filter <- dS_maker_silene(dS_mar_no_filter, data_gff_silene, gene_filter_mar,"S.marizii")

Merging tables before plotting 

In [ ]:
dS_merged_artemia <- rbind(dS_sin, dS_urm)
dS_merged_silene <- rbind(dS_lat, dS_dic,dS_dio,dS_heu,dS_mar)
dS_merged_silene_no_filter <- rbind(dS_lat_no_filter, dS_dic_no_filter,dS_dio_no_filter,dS_heu_no_filter,dS_mar_no_filter)

#### 4. File regristration for computing change points (step done with Mathematica)

In [ ]:
#%use bash

mkdir /PATH/dS

In [ ]:
#%use R
# for_ML_dS_function <- function(data){
#    data1<-get(data)
#    data2 <- data1 %>% select(Start,dS,Length) %>% arrange(Start)
#    write_csv(data2,paste0("dS/",data,".csv"))
#    return()
#  }

# Kept commented to prevent overwriting existing files

# for_ML_dS_function("dS_sin")
# for_ML_dS_function("dS_urm")
# 
# for_ML_dS_function("dS_lat")
# for_ML_dS_function("dS_dic")
# for_ML_dS_function("dS_dio")
# for_ML_dS_function("dS_heu")
# for_ML_dS_function("dS_mar")
# 
# for_ML_dS_function("dS_lat_no_filter")
# for_ML_dS_function("dS_dic_no_filter")
# for_ML_dS_function("dS_dio_no_filter")
# for_ML_dS_function("dS_heu_no_filter")
# for_ML_dS_function("dS_mar_no_filter")

### II.4.c Change point analysis 

Change points were computed using a custom Wolfram script run via the terminal. An annotated version is available in the toolbox directory.


In [ ]:
%use bash 
export WOLFRAMSCRIPT_KERNELPATH="/Applications/Wolfram.app/Contents/MacOS/MathKernel" 
names=("sin" "urm" "lat" "dio" "dic" "heu" "mar")

cd /PATH/Analyses
for name in "${names[@]}"; do caffeinate -i ./change_point_recursive.sh cp_dS.wls dS/ "${name}"; done

for f in dS/topo0_cutted_*.csv; do echo -e "NA\nNA,NA\nNA,NA" > "$f"; echo "  remplacé : $f" ; done
for name in "${names[@]}"; do bash process_topo.sh "${name}" dS/; done

cd dS
for f in topo*; do   mv "$f" "${f/topo/dS}"; done


### II.4.d Plots

#### 1. without change points 

In [ ]:
plot_dS_artemia<- ggplot(dS_merged_artemia, aes(x = Middle, y = dS)) +
  geom_point(size = 1) +
  xlab("position (pdb)") +
  ylab("dS") +
  #geom_vline(xintercept = c(48638, 5769054, 10069580, 15830480, 36776284), 
             #linetype = "dashed", color = "blue") +
  facet_grid(Species ~ ., scales = "free_x") +
  xlim(0,110000000) +
  theme_bw() #+

plot_dS_silene<- ggplot(dS_merged_silene, aes(x = Middle, y = dS)) +
  geom_point(size = 1) +
  xlab("position (pdb)") +
  ylab("dS") +
  #geom_vline(xintercept = c(48638, 5769054, 10069580, 15830480, 36776284), 
             #linetype = "dashed", color = "blue") +
  facet_grid(Species ~ ., scales = "free_x") +
  xlim(0,350000000) +
  ylim(0, 0.2) +
  theme_bw() #+

plot_dS_silene_no_filter<- ggplot(dS_merged_silene_no_filter, aes(x = Middle, y = dS)) +
  geom_point(size = 1) +
  xlab("position (pdb)") +
  ylab("dS") +
  #geom_vline(xintercept = c(48638, 5769054, 10069580, 15830480, 36776284), 
             #linetype = "dashed", color = "blue") +
  facet_grid(Species ~ ., scales = "free_x") +
  xlim(0,350000000) +
  ylim(0, 0.2) +
  theme_bw() #+


plot_dS_artemia
plot_dS_silene
plot_dS_silene_no_filter

#### 2. With change points 

Data importation 

In [ ]:
for (name in names) {
  tmp <- read.csv(paste0("dS/dS_cutted_",name,"_errors_final.csv"),header=FALSE)
  tmp<- tmp[order(tmp[,1]), , drop = FALSE]
 assign(paste0("cp_dS_", name,"_errors_final"), tmp)
 assign(paste0("cp_dS_", name,"_final"), sort(unlist(as.numeric(read.csv(paste0("dS/dS_cutted_",name,"_final.csv"),header=FALSE)))))
}

For Silene, a manual axis break was added to improve readability


In [ ]:
Gap_start = 101e6
Gap_end = 226e6

# limits found in the litterature
arrows <- data.frame(x = c(5769054, 10069580, 15830480, 36776284, 315567532), y = c(0,0,0,0,0), yend = c(0.1,0.1,0.1,0.1,0.1))

# Compress genomic coordinates to remove the gap region
compress_x <- function(x, gap_start = Gap_start, gap_end = Gap_end) {
  y <- x
  idx <- !is.na(x) & x >= gap_end
  y[idx] <- x[idx] - (gap_end - gap_start)
  y
}

# Compute mean dS within a genomic interval
mean_function <- function(df, limit1, limit2, pair) {
  bloc <- df %>% filter(Start >= limit1 & Start < limit2) %>% summarise(mean = mean(dS)) %>% mutate(lim1 = limit1, lim2 = limit2)
  return(bloc)
}

# Split blocks overlapping the gap and apply coordinate compression
process_blocs <- function(b_list, gap_start = Gap_start, gap_end = Gap_end) {
  new_b_list <- list()
  for (b in b_list) {
    temp <- b
    expanded <- lapply(seq_len(nrow(temp)), function(i) {
      row <- temp[i, ]
      if (row$lim1 < gap_start && row$lim2 > gap_start) {
        row_before <- row; row_before$lim2 <- gap_start
        row_after <- row; row_after$lim1 <- gap_end
        rbind(row_before, row_after)
      } else { row }
    })
    new_b_list <- c(new_b_list, expanded)
  }
  final_b <- do.call(rbind, new_b_list)
  final_b$lim1 <- compress_x(final_b$lim1)
  final_b$lim2 <- compress_x(final_b$lim2)
  return(final_b)
}

# Plot dS along chromosome with axis break and change points
plot_dS_silene_function_subset <- function(xlim1, xlim2, cp_dS_final, cp_dS_errors_final, dS_data, b_list, gap_start = Gap_start, gap_end = Gap_end) {

  b_df <- process_blocs(b_list, gap_start, gap_end)

  x_breaks_real <- seq(from = ceiling(xlim1/2.5e7)*2.5e7, to = floor(xlim2/2.5e7)*2.5e7, by = 2.5e7)
  x_breaks_real <- x_breaks_real[!(x_breaks_real > gap_start & x_breaks_real < gap_end)]
  x_breaks_compressed <- compress_x(x_breaks_real)
  x_labels <- scales::scientific(x_breaks_real)

  dS_points <- filter(dS_data, Start < gap_start | Start >= gap_end)

  plot <- ggplot() +
    geom_point(data = dS_points, aes(x = compress_x(Start), y = dS), size = log(dS_points$Length)/7, alpha = log(dS_points$Length)/15, shape = 16) +
    geom_segment(data = b_df, aes(x = lim1, xend = lim2, y = mean, yend = mean)) +
    xlab("Position (bp)") + ylab("dS") + ylim(0,0.2) +
    geom_vline(xintercept = compress_x(cp_dS_final), linetype = "dashed", color = "darkgray") +
    geom_rect(data = cp_dS_errors_final, aes(xmin = compress_x(V1), xmax = compress_x(V2), ymin = -Inf, ymax = Inf), inherit.aes = FALSE, fill = "darkgray", alpha = 0.2) +
    theme_bw() + theme(legend.position = "none") + facet_grid(vars(Species)) +
    scale_x_continuous(breaks = x_breaks_compressed, labels = x_labels, expand = c(0,0)) +
    coord_cartesian(xlim = compress_x(c(xlim1, xlim2))) +
    annotate("segment", x = gap_start - 2e6, xend = gap_start + 2e6, y = 0, yend = 0.005, size = 1) +
    annotate("segment", x = gap_start - 2e6, xend = gap_start + 2e6, y = 0.005, yend = 0, size = 1)

  return(plot)
}

# Combine all species pairs into stacked plot
combined_dS_plot_silene_function_subset <- function(xlim1, xlim2) {

  plot_list <- list()

  for (i in seq_along(names[-c(1,2)])) {

    name <- names[-c(1,2)][i]
    dS_data <- get(paste0("dS_", name))
    cp_dS_final <- get(paste0("cp_dS_", name, "_final"))
    cp_dS_errors_final <- get(paste0("cp_dS_", name, "_errors_final"))

    cp <- sort(cp_dS_final)
    starts <- c(0, cp)
    ends <- c(cp, last(sort(dS_data$Start))+1)

    b_list <- Map(function(s,e) { mean_function(dS_data, s, e, pair) }, starts, ends)

    plot <- plot_dS_silene_function_subset(xlim1, xlim2, cp_dS_final, cp_dS_errors_final, dS_data, b_list)

    if (i < length(names[-c(1,2)])) {
      plot <- plot + theme(axis.title.x = element_blank(), axis.text.x = element_blank(), axis.ticks.x = element_blank())
    } else {
      plot <- plot + geom_point(data = arrows, aes(x = x, y = y), shape = 17, size = 2, color = "blue", inherit.aes = FALSE)
    }

    plot_list[[name]] <- plot
  }

  plot_zoom <- wrap_plots(plot_list, ncol = 1) + plot_layout(guides = "collect") &
    theme(legend.position = "none", panel.background = element_rect(fill = NA), plot.background = element_rect(fill = NA))

  return(plot_zoom)
}

plot_dS_silene_total <- combined_dS_plot_silene_function_subset(0, 350000000)


In [ ]:
plot_dS_silene_total

For Artemia, no need to add an axis break 

In [ ]:
# limits found in the litterature
arrows_artemia <- data.frame(
  x = c(35322500, 63525001, 88115001,100585001),
  y = c(0, 0, 0, 0),
  yend = c(0.1, 0.1, 0.1,0.1)
)

# Plot dS along chromosome 
plot_dS_artemia_function_subset <- function(xlim1, xlim2, cp_dS_final, cp_dS_errors_final, dS_data, b_list, gap_start = Gap_start, gap_end = Gap_end) {

  b_df <- do.call(rbind, b_list)

  plot <- ggplot() +
   geom_point(data = dS_data, aes(x = compress_x(Start), y = dS), size = log(dS_data$Length)/7, alpha = log(dS_data$Length)/15, shape = 16) +
   geom_segment(data = b_df, aes(x = lim1, xend = lim2, y = mean, yend = mean)) +
   xlab("Position (bp)") +
   ylab("dS") +
   ylim(0,0.2) +
   geom_vline(xintercept = cp_dS_final, linetype = "dashed", color = "darkgray") +
   geom_rect(data = cp_dS_errors_final, aes(xmin = V1, xmax = V2, ymin = -Inf, ymax = Inf), inherit.aes = FALSE, fill = "darkgray", alpha = 0.2) +
   theme_bw() +
   theme(legend.position = "none") +
   facet_grid(vars(Species))  # Facet by species to display each comparison separately

  return(plot)
}

combined_dS_plot_artemia_function_subset <- function(xlim1, xlim2) {

  plot_list <- list()

  for(i in seq_along(names[c(1,2)])) {

    name <- names[c(1,2)][i]

    dS_data <- get(paste0("dS_", name))
    cp_dS_final <- get(paste0("cp_dS_", name,"_final"))
    cp_dS_errors_final <- get(paste0("cp_dS_", name, "_errors_final"))

    # Define genomic blocks based on change points
    cp <- sort(cp_dS_final)
    starts <- c(0, cp)
    ends <- c(cp, last(sort(dS_data$Start))+1)

    # Compute mean dS for each block
    b_list <- Map(function(s,e) {
      bloc <- mean_function(dS_data, s, e, pair)
      bloc$lim1 <- bloc$lim1
      bloc$lim2 <- bloc$lim2
      bloc
    }, starts, ends)

    # Generate plot for the current dataset
    plot <- plot_dS_artemia_function_subset(xlim1 = xlim1, xlim2 = xlim2, cp_dS_final = cp_dS_final, cp_dS_errors_final = cp_dS_errors_final, dS_data = dS_data, b_list = b_list)

    # Remove x-axis elements for consistent stacked visualization
    plot <- plot + theme(axis.title.x = element_blank(), axis.text.x = element_blank(), axis.ticks.x = element_blank())

    plot_list[[name]] <- plot

  }

  # Combine plots vertically. 
  plot_zoom <- wrap_plots(plot_list, ncol = 1) + plot_layout(guides = "collect") &
    theme(legend.position = "none") +
    theme(panel.background = element_rect(fill = NA), plot.background = element_rect(fill = NA))

  return(plot_list) 
    # This would normally return(plot_zoom), but was modified for the final figure to combine topology and dS results for Artemia. 
    # The original implementation was retained for potential future use.

}

plot_dS_artemia_total <- combined_dS_plot_artemia_function_subset(0, 350000000)


In [ ]:
plot_dS_artemia_total

## II.5 Working on topologies

### II.5.1 Data importation 

#### 1. Gene list files

In [ ]:
#%use R
for (pair in pairs) {
  assign(
    paste0("list_gen_", pair),
    read.table(
      paste0("list_gene/gene_commun_", pair, "_corrected.txt"),
      header = FALSE, col.names = c("1"), sep = "\t", stringsAsFactors = FALSE, comment.char = ""))
}

for (pair in pairs[-1]) {
  assign(
    paste0("list_gen_", pair,"_filtered"),
    read.table(
      paste0("list_gene/gene_commun_", pair, "_filtered_corrected.txt"),
      header = FALSE, col.names = c("1"), sep = "\t", stringsAsFactors = FALSE, comment.char = ""))
}

#### 2. Duplicate SNPs info 

In [ ]:
snp_duplicate_importation_function<-function(filename,data_gtf) {
  data<-read_csv(filename, col_names = TRUE, show_col_types = FALSE)
  names(data)<- c("ID","prop")
  data <- data %>% left_join(select(data_gtf, ID, Start), by=join_by("ID")) 
}

prop_duplicate_sin <- snp_duplicate_importation_function("snp_duplicate/prop_snp_duplicate_sinica.csv",data_gtf_artemia)
prop_duplicate_urm <- snp_duplicate_importation_function("snp_duplicate/prop_snp_duplicate_urmiana.csv",data_gtf_artemia)

prop_duplicate_lat <- snp_duplicate_importation_function("snp_duplicate/prop_snp_duplicate_latifolia.csv", data_gff_silene)
prop_duplicate_dic <- snp_duplicate_importation_function("snp_duplicate/prop_snp_duplicate_diclinis.csv", data_gff_silene)
prop_duplicate_dio <- snp_duplicate_importation_function("snp_duplicate/prop_snp_duplicate_dioica.csv", data_gff_silene)
prop_duplicate_heu <- snp_duplicate_importation_function("snp_duplicate/prop_snp_duplicate_heuffelii.csv", data_gff_silene)
prop_duplicate_mar <- snp_duplicate_importation_function("snp_duplicate/prop_snp_duplicate_marizii.csv", data_gff_silene)

#### 3. Topologies

In [ ]:
topo_importation_function<-function(filename) {
  df<-read_csv(filename, col_names = TRUE, show_col_types = FALSE)
  names(df)[grepl("^0\\d{4}$", names(df))] <- sub("^0", "", names(df)[grepl("^0\\d{4}$", names(df))])
  return(df)
}

for (pair in pairs) {
  assign(
    paste0("topo_", pair),
    topo_importation_function(paste0("topo/topo_", pair, ".csv")))
}

for (pair in pairs[-1]) {
  assign(
    paste0("topo_", pair,"_filtered"),
    topo_importation_function(paste0("topo/topo_", pair, "_filtered.csv")))
}

### II.5.b Data treatment

#### 1. General treatment 

In [ ]:
# Define desired order of topologies for consistent column arrangement
desired_order <- c("1010","0101","0100","0001","1000","0010","1100","0011","1110","1101","1011","0111","1001","0110")

# Process raw topology counts: normalize, remove duplicates, keep longest transcript per gene
topo_traitement_function <- function(topologies, list_gene, data_duplicate1, data_duplicate2, data_gtf) {
  data_duplicate1 <- filter(data_duplicate1, prop >= 0.25)
  data_duplicate2 <- filter(data_duplicate2, prop >= 0.25)
  data_duplicate <- merge(data_duplicate1, data_duplicate2, by = c("ID","Start"), all = TRUE) 
  
  # Remove uninformative topologies (0000 and 1111)
  topologies_no_01111 <- as.data.frame(topologies) %>% select(-"0000", -"1111") 
  
  # Normalize to proportions
  topologies_normalized <- apply(topologies_no_01111, 1, function(row) row / sum(row))
  topologies_normalized <- replace(topologies_normalized, is.na(topologies_normalized) | is.nan(topologies_normalized), 0)
  topologies_normalized <- as.data.frame(t(topologies_normalized))
  
  # Combine with gene IDs and annotate chromosome positions
  topo <- cbind(list_gene$X1, topologies_normalized)
  names(topo)[1] <- "ID"
  topo <- left_join(topo, select(data_gtf, ID, Chromosome, Start, End), by = join_by(ID)) %>%
    filter(!ID %in% data_duplicate$ID) %>%                 # remove duplicated SNPs
    filter(!if_all(.cols = 2:15, .fns = ~ . == 0)) %>%    # remove rows with no topology info
    mutate(Length = End - Start + 1) %>%
    select(-End) %>%
    relocate((ncol(.)-2):ncol(.), .before = 1) %>%
    mutate(GeneID = str_remove(ID, "\\.t\\d+$")) %>%
    group_by(GeneID) %>%
    filter(Length == max(Length)) %>%                     # keep only longest transcript
    slice_sample(n = 1) %>%                               # randomly pick if tie
    ungroup() %>%
    select(-GeneID)
  
  topo <- topo %>% relocate(all_of(desired_order), .after = "ID")
  return(topo)
}

# Convert processed topology dataframe to long format with pair info
topo_traitement2_function <- function(topologie, pair, data_duplicate1, data_duplicate2, data_gtf) {
  topologie <- melt(topologie, id.vars = c("ID", "Start", "Chromosome", "Length"))
  topologie <- mutate(topologie, Pair = pair)
  topologie$Pair <- as.factor(topologie$Pair)
  return(topologie)
}


For Artemia 

In [ ]:
# treatment of the weird gene manually
g1385.g1 <- data.frame(Chromosome = "Chromosome_1", Start = 88970358, Length = 18716, ID = "g1385.g1",
  "1010" = 0, "0101" = 0, "0100" = 0, "0001" = 2/9, "1000" = 0, "0010" = 2/9, "1100" = 2/9,
  "0011" = 2/9, "1110" = 0, "1101" = 1/9, "1011" = 0, "0111" = 0, "1001" = 0, "0110" = 0, check.names = FALSE)

g1385.g2 <- data.frame(Chromosome = "Chromosome_1", Start = 89005175, Length = 18027, ID = "g1385.g2",
  "1010" = 2/14, "0101" = 6/14, "0100" = 3/14, "0001" = 2/14, "1000" = 1/14, "0010" = 0,
  "1100" = 0, "0011" = 0, "1110" = 0, "1101" = 0, "1011" = 0, "0111" = 0, "1001" = 0, "0110" = 0, check.names = FALSE)

# Process Sinica-Urmania pair and filter for Chromosome_1 region of interest
topo_sin_urm_traited <- topo_traitement2_function(
  bind_rows(topo_traitement_function(topo_sin_urm, list_gen_sin_urm, prop_duplicate_sin, prop_duplicate_urm, data_gtf_artemia), g1385.g1, g1385.g2),
  "sin_urm"
) %>% filter(Chromosome == "Chromosome_1") %>% filter(Start >= 34218781)


For Silene

In [ ]:
# Process all Silene pairs for chrX
for (pair in pairs[-1]) {
  espece1 <- strsplit(pair, "_")[[1]][1]
  espece2 <- strsplit(pair, "_")[[1]][2]
  topo_input <- get(paste0("topo_", pair))
  list_input <- get(paste0("list_gen_", pair))
  prop1 <- get(paste0("prop_duplicate_", espece1))
  prop2 <- get(paste0("prop_duplicate_", espece2))
  
  result <- topo_traitement2_function(topo_traitement_function(topo_input, list_input, prop1, prop2, data_gff_silene), pair) %>% filter(Chromosome == "chrX")
  assign(paste0("topo_", pair, "_traited"), result)
}

# Repeat for filtered topologies
for (pair in pairs[-1]) {
  espece1 <- strsplit(pair, "_")[[1]][1]
  espece2 <- strsplit(pair, "_")[[1]][2]
  topo_input <- get(paste0("topo_", pair, "_filtered"))
  list_input <- get(paste0("list_gen_", pair, "_filtered"))
  prop1 <- get(paste0("prop_duplicate_", espece1))
  prop2 <- get(paste0("prop_duplicate_", espece2))
  
  result <- topo_traitement2_function(topo_traitement_function(topo_input, list_input, prop1, prop2, data_gff_silene), pair) %>% filter(Chromosome == "chrX")
  assign(paste0("topo_", pair, "_traited_filtered"), result)
}


Combine all processed pairs into a single dataframe

In [ ]:
topo_silene <- do.call(rbind, lapply(pairs[-1], function(pair) get(paste0("topo_", pair, "_traited"))))
topo_silene_filtered <- do.call(rbind, lapply(pairs[-1], function(pair) get(paste0("topo_", pair, "_traited_filtered"))))


#### 2. Treatment for change point analysis 

In [ ]:
%use bash 

mkdir /PATH/Topologies_all_topo

In [ ]:
#%use R
# Process topologies for control points (CP): remove duplicates, keep longest transcript, normalize
topo_traitement_function_for_cp <- function(topologies, list_gene, data_duplicate1, data_duplicate2, data_gtf) {
  data_duplicate1 <- filter(data_duplicate1, prop >= 0.25)
  data_duplicate2 <- filter(data_duplicate2, prop >= 0.25)
  data_duplicate <- merge(data_duplicate1, data_duplicate2, by = c("ID","Start"), all = TRUE) 
  
  # Remove uninformative topologies
  topologies_no_01111 <- as.data.frame(topologies) %>% select(-"0000", -"1111")
  
  # Combine with gene IDs and annotate chromosome positions
  topo <- cbind(list_gene$X1, as.data.frame(topologies_no_01111))
  names(topo)[1] <- "ID"
  topo <- left_join(topo, select(data_gtf, ID, Chromosome, Start, End), by = join_by(ID)) %>%
    filter(!ID %in% data_duplicate$ID) %>%                    # remove duplicated SNPs
    filter(!if_all(.cols = 2:15, .fns = ~ . == 0)) %>%       # remove rows with no topology info
    mutate(Length = End - Start + 1) %>%
    select(-End) %>%
    relocate((ncol(.)-2):ncol(.), .before = 1) %>%
    mutate(GeneID = str_remove(ID, "\\.t\\d+$")) %>%
    group_by(GeneID) %>%
    filter(Length == max(Length)) %>%                         # keep only longest transcript
    slice_sample(n = 1) %>%                                   # randomly pick if tie
    ungroup() %>%
    select(-GeneID)
  
  topo <- topo %>% relocate(all_of(desired_order), .after = "ID")
  return(topo)
}

# Process Sinica-Urmania pair for CP
topo_sin_urm_cp <- topo_traitement_function_for_cp(topo_sin_urm, list_gen_sin_urm, prop_duplicate_sin, prop_duplicate_urm, data_gtf_artemia) %>%
  filter(Chromosome == "Chromosome_1")

# Process all Silene pairs for chrX (raw data)
for (pair in pairs[-1]) {
  espece1 <- strsplit(pair, "_")[[1]][1]
  espece2 <- strsplit(pair, "_")[[1]][2]
  topo_input <- get(paste0("topo_", pair))
  list_input <- get(paste0("list_gen_", pair))
  prop1 <- get(paste0("prop_duplicate_", espece1))
  prop2 <- get(paste0("prop_duplicate_", espece2))
  
  result <- topo_traitement_function_for_cp(topo_input, list_input, prop1, prop2, data_gff_silene) %>%
    filter(Chromosome == "chrX") %>% 
    filter(!(ID %in% c("chr12_001954.1","chr12_001860.1","chr12_001912.1"))) # remove problematic genes
  assign(paste0("topo_", pair, "_cp"), result)
}

# Process all Silene pairs for chrX (filtered data)
for (pair in pairs[-1]) {
  espece1 <- strsplit(pair, "_")[[1]][1]
  espece2 <- strsplit(pair, "_")[[1]][2]
  topo_input <- get(paste0("topo_", pair, "_filtered"))
  list_input <- get(paste0("list_gen_", pair, "_filtered"))
  prop1 <- get(paste0("prop_duplicate_", espece1))
  prop2 <- get(paste0("prop_duplicate_", espece2))
  
  result <- topo_traitement_function_for_cp(topo_input, list_input, prop1, prop2, data_gff_silene) %>%
    filter(Chromosome == "chrX") %>% 
    filter(!(ID %in% c("chr12_001954.1","chr12_001860.1","chr12_001912.1")))
  assign(paste0("topo_", pair, "_cp_filtered"), result)
}


File regristration for computing change points (step done with Mathematica)

In [ ]:
# Kept commented to prevent overwriting existing files
# write_csv(filter(topo_sin_urm_cp,Start < 88970358),"Topologies_all_topo/sin_urm_topo_limits.csv")
#
# write_csv(topo_lat_dio_cp,"Topologies/lat_dio_topo_limits.csv")
# write_csv(topo_lat_dic_cp, "Topologies/lat_dic_topo_limits.csv")
# write_csv(topo_lat_heu_cp, "Topologies/lat_heu_topo_limits.csv")
# write_csv(topo_lat_mar_cp, "Topologies/lat_mar_topo_limits.csv")
# write_csv(topo_dio_dic_cp, "Topologies/dio_dic_topo_limits.csv")
# write_csv(topo_dio_heu_cp, "Topologies/dio_heu_topo_limits.csv")
# write_csv(topo_dio_mar_cp, "Topologies/dio_mar_topo_limits.csv")
# write_csv(topo_dic_heu_cp, "Topologies/dic_heu_topo_limits.csv")
# write_csv(topo_dic_mar_cp, "Topologies/dic_mar_topo_limits.csv")
# write_csv(topo_heu_mar_cp, "Topologies/heu_mar_topo_limits.csv")
# 
# write_csv(topo_lat_dio_cp_filtered,"Topologies_all_topo/lat_dio_topo_limits.csv")
# write_csv(topo_lat_dic_cp_filtered, "Topologies_all_topo/lat_dic_topo_limits.csv")
# write_csv(topo_lat_heu_cp_filtered, "Topologies_all_topo/lat_heu_topo_limits.csv")
# write_csv(topo_lat_mar_cp_filtered, "Topologies_all_topo/lat_mar_topo_limits.csv")
# write_csv(topo_dio_dic_cp_filtered, "Topologies_all_topo/dio_dic_topo_limits.csv")
# write_csv(topo_dio_heu_cp_filtered, "Topologies_all_topo/dio_heu_topo_limits.csv")
# write_csv(topo_dio_mar_cp_filtered, "Topologies_all_topo/dio_mar_topo_limits.csv")
# write_csv(topo_dic_heu_cp_filtered, "Topologies_all_topo/dic_heu_topo_limits.csv")
# write_csv(topo_dic_mar_cp_filtered, "Topologies_all_topo/dic_mar_topo_limits.csv")
# write_csv(topo_heu_mar_cp_filtered, "Topologies_all_topo/heu_mar_topo_limits.csv")

#### 3. Smooth function for plot without change points 

In [ ]:
# Smooth topology proportions using rolling mean
smooth_function <- function(topo, list_gene, data_duplicate1, data_duplicate2, data_gtf) {
  topo <- topo_traitement_function(topo, list_gene, data_duplicate1, data_duplicate2, data_gtf)
  topo_without_id_start <- topo %>% select(-ID, -Start, -Chromosome, -Length)
  
  # Apply rolling mean smoothing (window = 3)
  topo_smoothed <- apply(topo_without_id_start, 2, function(x) rollmean(x, k = 3, fill = NA))
  topo_smoothed <- as.data.frame(topo_smoothed) 
  topo_smoothed <- mutate_at(topo_smoothed, vars(), as.numeric)
  
  # Reattach identifiers
  topo_smoothed <- cbind(ID = topo$ID, Start = topo$Start, Chromosome = topo$Chromosome, Length = topo$Length, topo_smoothed)
  return(topo_smoothed)
}

# Process Sinica-Urmania pair
topo_sin_urm_smooth <- topo_traitement2_function(
  smooth_function(topo_sin_urm, list_gen_sin_urm, prop_duplicate_sin, prop_duplicate_urm, data_gtf_artemia),
  "sin_urm"
)

# Smooth all Silene pairs (raw)
for (pair in pairs[-1]) {
  espece1 <- strsplit(pair, "_")[[1]][1]
  espece2 <- strsplit(pair, "_")[[1]][2]
  topo_obj <- get(paste0("topo_", pair))
  list_gene <- get(paste0("list_gen_", pair))
  dup1 <- get(paste0("prop_duplicate_", espece1))
  dup2 <- get(paste0("prop_duplicate_", espece2))
  
  result <- topo_traitement2_function(
    smooth_function(topo_obj, list_gene, dup1, dup2, data_gff_silene),
    pair
  )
  assign(paste0("topo_", pair, "_smooth"), result)
}

# Smooth all Silene pairs (filtered)
for (pair in pairs[-1]) {
  espece1 <- strsplit(pair, "_")[[1]][1]
  espece2 <- strsplit(pair, "_")[[1]][2]
  topo_obj <- get(paste0("topo_", pair, "_filtered"))
  list_gene <- get(paste0("list_gen_", pair, "_filtered"))
  dup1 <- get(paste0("prop_duplicate_", espece1))
  dup2 <- get(paste0("prop_duplicate_", espece2))
  
  result <- topo_traitement2_function(
    smooth_function(topo_obj, list_gene, dup1, dup2, data_gff_silene),
    pair
  )
  assign(paste0("topo_", pair, "_smooth_filtered"), result)
}

# Combine all smoothed Silene pairs
topo_silene_smooth <- do.call(rbind, lapply(pairs[-1], function(pair) get(paste0("topo_", pair, "_smooth"))))
topo_silene_smooth_filtered <- do.call(rbind, lapply(pairs[-1], function(pair) get(paste0("topo_", pair, "_smooth_filtered"))))


### II.5.c Change point analysis 

In [ ]:
%use bash 
export WOLFRAMSCRIPT_KERNELPATH="/Applications/Wolfram.app/Contents/MacOS/MathKernel" 
pairs=("sin_urm" "lat_dio" "lat_dic" "lat_heu" "lat_mar" "dio_dic" "dio_heu" "dio_mar" "dic_heu" "dic_mar" "heu_mar")

cd /PATH/Analyses
for pair in "${pairs[@]}"; do     caffeinate -i ./change_point_recursive.sh ./cp.wls Topologies_all_topo/ "${pair}"; done

for f in Topologies_all_topo/topo0_cutted_*.csv; do echo -e "NA\nNA,NA\nNA,NA" > "$f"; echo "  remplacé : $f" ; done
for pair in "${pairs[@]}"; do     bash process_topo.sh "${pair}" Topologies_all_topo/; done


### II.5.d Plots

#### 1. Plot without change point 

In [ ]:
plot_topo_function <- function(xlim1,xlim2,topo,topo_smooth) {
  plot_topo <- ggplot()+
    geom_line(data = topo_smooth, aes(x = Start, y = value, color = variable), linewidth = 0.7) +
    geom_point(data=topo, aes(x = Start, y = value, color=variable,size=sqrt(Length)/50,alpha=sqrt(Length)/50),shape=16) +
    scale_size_identity()+
    xlab("position (pdb)") +
    ylab("nbr_topo") + 
    scale_x_continuous(limits = c(xlim1, xlim2),expand = c(0, 0)) +
    ylim(0,1) +
    scale_fill_manual(values = c( "#FF0000", "chartreuse3", "#FF00DB" , "#FF6D00" , "#0024FF"," darkred" , "#FFDB00" ,"turquoise3"," darkred" , "#B600FF" ," darkred"," darkred")) +
    facet_grid(vars(Pair)) +
    theme_bw() #+
  return(plot_topo)
}

T1<-plot_topo_function(0,350000000,topo_silene,topo_silene_smooth)
T2<-plot_topo_function(0,350000000,topo_silene_filtered,topo_silene_smooth_filtered)
T3<-plot_topo_function(0,110000000,topo_sin_urm_traited,topo_sin_urm_smooth)
T1
T2
T3

#### 2. Plot with change point

Data importation 

In [ ]:
for (pair in pairs) {
  tmp <- read.csv(paste0("Topologies_all_topo/topo_cutted_",pair,"_errors_final.csv"),header=FALSE)
  tmp<- tmp[order(tmp[,1]), , drop = FALSE]
 assign(paste0("cp_topo_", pair,"_errors_final"), tmp)
 assign(paste0("cp_topo_", pair,"_final"), sort(unlist(as.numeric(read.csv(paste0("Topologies_all_topo/topo_cutted_",pair,"_final.csv"),header=FALSE)))))
}

For Silene, a manual axis break was added to improve readability


In [ ]:
Gap_start =71e6
Gap_end = 251e6

# limits of the litterature
arrows <- data.frame(
  x = c(5769054, 10069580, 15830480,36776284,315567532),
  y = c(0, 0, 0,0,0),
  yend = c(0.2, 0.2, 0.2,0.2,0.2)
)

# Compress X-axis values to create a gap
compress_x <- function(x, gap_start = Gap_start, gap_end = Gap_end) {
  y <- x
  # Ne toucher que les valeurs non NA >= gap_end
  idx <- !is.na(x) & x >= gap_end
  y[idx] <- x[idx] - (gap_end - gap_start)
  y
}

# Compute proportions of topologies within a block
prop_function <- function(df, limit1, limit2, pair) {
  bloc <- df %>%
    filter(Start >= limit1 & Start < limit2) %>%
    summarise(across(`1010`:`0110`, sum, na.rm = TRUE)) %>%
    mutate(across(everything(), ~ .x / sum(c_across(everything())))) %>%
    pivot_longer(
      cols = everything(),
      names_to = "variable",
      values_to = "proportion"
    ) %>%
    mutate(
      cum_low  = c(0, cumsum(proportion)[-length(proportion)]),
      cum_high = cumsum(proportion),
      lim1 = limit1,
      lim2 = limit2
    )

  return(bloc)
}

# Split blocks across gap
process_blocs <- function(b_list, gap_start = Gap_start, gap_end = Gap_end) {
  new_b_list <- list()
  
  for(b in b_list) {
    temp <- b
    expanded <- lapply(seq_len(nrow(temp)), function(i) {
      row <- temp[i, ]
      if(row$lim1 < gap_start && row$lim2 > gap_start) {
        row_before <- row
        row_before$lim2 <- gap_start
        row_after <- row
        row_after$lim1 <- gap_end
        row_after$lim2 <- row$lim2
        rbind(row_before, row_after)
      } else {
        row
      }
    })
    new_b_list <- c(new_b_list, expanded)
  }
  
  final_b <- do.call(rbind, new_b_list)
  final_b$lim1 <- compress_x(final_b$lim1)
  final_b$lim2 <- compress_x(final_b$lim2)
  
  return(final_b)
}

# Plot a single topology subset
plot_topo_silene_function_subset <- function(xlim1, xlim2, cp_topo_final, cp_topo_errors_final, topo_traited_filtered, b_list, gap_start = Gap_start, gap_end = Gap_end) {
  b_df <- process_blocs(b_list, gap_start, gap_end)
  x_breaks_real <- seq(from = ceiling(xlim1/2.5e7)*2.5e7,
                       to   = floor(xlim2/2.5e7)*2.5e7,
                       by   = 2.5e7)
  x_breaks_real <- x_breaks_real[!(x_breaks_real > gap_start & x_breaks_real < gap_end)]
  x_breaks_compressed <- compress_x(x_breaks_real)
  x_labels <- scales::comma(x_breaks_real)
  
  plot <- ggplot(topo_traited_filtered) +
    geom_rect(
      data = b_df,
      aes(xmin = lim1, xmax = lim2, ymin = cum_low, ymax = cum_high, fill = variable),
      color = NA, alpha = 0.3, inherit.aes = FALSE
    ) +
    # change points 
    geom_vline(xintercept = compress_x(cp_topo_final), linetype = "dashed", color = "darkgray") +
    geom_rect(
      data = cp_topo_errors_final,
      aes(xmin = compress_x(V1), xmax = compress_x(V2), ymin = -Inf, ymax = Inf),
      inherit.aes = FALSE, fill = "darkgray", alpha = 0.2
    ) +
     scale_fill_manual(values = c( "#FFCC02", "#FF6D00", "#B600FF" , "#A9D400" , "#03FFFF","#4D4D4D","#784321" ,"#FF0000","black","#0024FF","#DE8787" , "#FF00DB" ,"#008001", "#019E9E")) +
    theme_bw() +
    theme(legend.position = "none") +
    labs(x = "Position (bp)",y = "Topologies proportion")+
    facet_grid(vars(Pair)) +
    scale_x_continuous(breaks = x_breaks_compressed, labels = x_labels, expand = c(0, 0)) +
    coord_cartesian(xlim = compress_x(c(xlim1, xlim2))) +
    # Small zigzag to indicate gap
    annotate("segment", x = gap_start - 1e6, xend = gap_start + 1e6, y = -0.03, yend = -0.01, size = 0.8) +
    annotate("segment", x = gap_start - 1e6, xend = gap_start + 1e6, y = -0.01, yend = -0.03, size = 0.8)
  summary(b_df$lim1); summary(b_df$lim2)
  return(plot)
}

# Combine all pairs into a single plot
combined_topo_plot_silene_function_subset <- function(xlim1, xlim2) {
  
  plot_list <- list()
  
  for(i in seq_along(pairs[-1])) {
    pair <- pairs[-1][i]
    
    topo_cp <- get(paste0("topo_", pair, "_cp_filtered"))
    topo_traited_filtered <- get(paste0("topo_", pair, "_traited_filtered"))
    cp_topo_final <-get(paste0("cp_topo_", pair,"_final"))
    cp_topo_errors_final <- get(paste0("cp_topo_", pair, "_errors_final"))
    
    cp <- sort(cp_topo_final)
    starts <- c(first(sort(topo_traited_filtered$Start)), cp)
    ends <- c(cp, last(sort(topo_traited_filtered$Start))+1)
    
    b_list <- Map(function(s,e) {
      bloc <- prop_function(topo_cp, s, e, pair)
      bloc$lim1 <- bloc$lim1
      bloc$lim2 <- bloc$lim2
      bloc
    }, starts, ends)
    
   # Save block summaries
    summary_table <- tibble(
      pair = pair,
      bloc = seq_along(b_list),
      lim1 = map_dbl(b_list, ~ unique(.x$lim1)),
      lim2 = map_dbl(b_list, ~ unique(.x$lim2)),
      proportions = map(b_list, ~ .x$proportion)
    )
    assign(paste0("summary_prop_", pair), summary_table, envir = .GlobalEnv)
    
    plot <- plot_topo_silene_function_subset(
      xlim1 = xlim1,
      xlim2 = xlim2,
      cp_topo_final = cp_topo_final,
      cp_topo_errors_final = cp_topo_errors_final,
      topo_traited_filtered = topo_traited_filtered,
      b_list = b_list
    ) 
    
    if(i < length(pairs[-1])) {
      plot <- plot + theme(axis.title.x = element_blank(),
                           axis.text.x  = element_blank(),
                           axis.ticks.x = element_blank())
    } else {
      plot <- plot +  geom_point(data = arrows,
             aes(x = compress_x(x), y = y),  # yend = hauteur où tu veux le triangle
             shape = 17, size = 2,
             color = "blue",
             inherit.aes = FALSE)
    }
    
    plot_list[[pair]] <- plot 
    
  }
  

  plot_zoom <- wrap_plots(plot_list, ncol = 1) +
    plot_layout(guides = "collect") &
     theme(panel.background = element_rect(fill = NA), theme(legend.position = "none"),
      plot.background = element_rect(fill = NA))
  
  return(plot_zoom)
}

plot_silene_total <- combined_topo_plot_silene_function_subset(0, 340000000)
plot_silene_total

For Artemia, no need to add an axis break 

In [ ]:
plot_artemia_function_subset <- function(xlim1, xlim2, cp_topo_final, cp_topo_errors_final,topo_traited, b_list, gap_start = Gap_start, gap_end = Gap_end) {
  b_df <- do.call(rbind, b_list)
  plot <- ggplot(topo_traited) +
     geom_rect(
      data = b_df,
      aes(xmin = lim1, xmax = lim2, ymin = cum_low, ymax = cum_high, fill = variable),
      color = NA, alpha = 0.3, inherit.aes = FALSE
    ) +

    geom_vline(xintercept = cp_topo_final, linetype = "dashed", color = "darkgray") +
    geom_rect(
      data = cp_topo_errors_final,
      aes(xmin =V1, xmax = V2, ymin = -Inf, ymax = Inf),
      inherit.aes = FALSE, fill = "darkgray", alpha = 0.2
    ) +
    geom_point(data = filter(topo_traited,ID %in% c("g1385.g1","g1385.g2","g1561.t1","g1562.t2")), aes(x=Start, y=value, color = variable), alpha = 0.3) +

    scale_fill_manual(values = c( "#FFCC02", "#FF6D00", "#B600FF" , "#A9D400" , "#03FFFF","#4D4D4D","#784321" ,"#FF0000","black","#0024FF","#DE8787" , "#FF00DB" ,"#008001", "#019E9E")) +
    scale_colour_manual(values = c( "#0024FF","#03FFFF", "#A9D400","#FFCC02","#FF0000", "#FF6D00", "#B600FF" , "#FF00DB" ,"#019E9E"  ,"#008001" , "#DE8787","#784321","#4D4D4D","black")) +
    theme_bw() +
    #theme(legend.position = "none") + # for final figures, legends were added manually
    labs(x = "Position (bp)",y = "Topologies proportion")+
    facet_grid(vars(Pair)) 
  return(plot)
}

cp_artemia <- sort(cp_topo_sin_urm_final)
starts_A <- c(sort(topo_sin_urm_cp$Start)[1], cp_artemia)
ends_A <- c(cp_artemia, 88970358+1) #c(cp_artemia,last(sort(topo_sin_urm_cp$Start))+1)
    
b_list_artemia <- Map(function(s,e) {
  bloc <- prop_function(filter(topo_sin_urm_cp, Start <88970358), s, e, pair)
  bloc$lim1 <- bloc$lim1
  bloc$lim2 <- bloc$lim2
  bloc
}, starts_A, ends_A)
  

summary_prop_sin_urm <- tibble(
  pair = "sin_urm",
  bloc = seq_along(b_list_artemia),
  lim1 = map_dbl(b_list_artemia, ~ unique(.x$lim1)),
  lim2 = map_dbl(b_list_artemia, ~ unique(.x$lim2)),
  proportions = map(b_list_artemia, ~ .x$proportion))
    
plot_artemia_total <- plot_artemia_function_subset(
  xlim1 = 0,
  xlim2 = 100000000,
  cp_topo_final = cp_topo_sin_urm_final,
  cp_topo_errors_final = cp_topo_sin_urm_errors_final,
  topo_traited = topo_sin_urm_traited,
  b_list = b_list_artemia
) + scale_x_continuous(limits = c(0, 100800000)) +
  geom_point(data = arrows_artemia, aes(x = x, y = y),  # yend = hauteur où tu veux le triangle
         shape = 17, size = 2,
         color = "blue",
         inherit.aes = FALSE) 

plot_artemia_total

### II.5.e Working on Outliers 

#### 1. Silene

In [ ]:
# Find the block index in which a given gene position falls
find_bloc <- function(pos, limits) {
  which(pos >= limits$lim1 & pos < limits$lim2)
}

# Compute empirical p-value for observed gene topology given multinomial draws
prob_bloc_function <- function(n, p, obs, nb_tirage) {
  eps = 0.0000000001
  tirages <- replicate(nb_tirage, rmultinom(1, size = n, prob = p), simplify = FALSE) 
  log_tirages <- sapply(tirages, function(x) sum(x * log(p + eps))) 
  log_obs <- sum(obs * log(p + eps)) # log-likelihood of observed counts
  sum(log_tirages < log_obs)/nb_tirage # fraction of draws with log-likelihood < observed
}

# Compute p-values for all genes in each pair
for(i in seq_along(pairs[-1])) {
  pair <- pairs[-1][i]
  # pair <- "heu_mar"
  # Récupération des données
  summary_prop <- get(paste0("summary_prop_", pair))
  topo_cp <- get(paste0("topo_", pair, "_cp_filtered"))
  
  result <- topo_cp %>%
    rowwise() %>% mutate(
      # observation = vecteur x
      obs = list(c_across(`1010`:`0110`)),
      # bloc dans lequel tombe le gène
      bloc = find_bloc(Start, summary_prop),
      lim1 = summary_prop$lim1[bloc],
      lim2 = summary_prop$lim2[bloc],
      prob_bloc = list({
      p <- unlist(summary_prop$proportions[[bloc]])
      p <- p + c(0,0,0,0,0,0,0,0, 0.01,0.01,0.01,0.01, 0,0)
      p / sum(p)   
    }),
      size = sum(unlist(obs)),
      !!paste0("pvalue_", pair) := log(prob_bloc_function(size, unlist(prob_bloc), obs, 500000) + 1/500000)

    ) %>%
    ungroup() %>%
    select(Chromosome, ID, Start, Length, bloc, lim1, lim2, prob_bloc, obs, size, !!paste0("pvalue_", pair)) 

  assign(paste0("topo_", pair,"_cp_outliers"), result, envir = .GlobalEnv)
}

# Generate histograms and filter outliers per pair
plots <- list()
plots_hist <- list()

for (i in 1:10) {
  pair<- pairs[i+1]
  pair_name<- pairs_name[i+1]
  df_name <- paste0("topo_", pair, "_cp_outliers")
  df <- get(df_name)
  col_name <- paste0("pvalue_", pair)
  
  if (pair %in% c("lat_mar","dio_mar","dic_heu")) {
      seuil <- -6
    } else if (pair %in% c("lat_heu","heu_mar")) {
      seuil <- -6.2
    } else if (pair == "lat_dic") {
      seuil <-  -6.5
    } else if (pair %in% c("lat_dio","dic_mar")) {
      seuil <-  -7
    } else if (pair == "dio_dic") {
      seuil <-  -7.5
    } else if (pair == "dio_heu") {
      seuil <-  -5.8
    } 
  plot_hist <-ggplot(df, aes(x = .data[[col_name]])) +
    geom_histogram(bins = 50, fill = "grey70", color = "black") +
    geom_vline(xintercept = seuil, color = "darkblue", linetype = "dashed", linewidth = 1) +
    ggtitle(pair_name) +
    xlab("pvalue") +
    theme_bw()
  
  list_outlier <- df %>% filter(.data[[col_name]] < seuil)
  topo<- get(paste0("topo_", pair, "_cp_filtered"))
  new_topo_cp <- filter(topo,!(ID%in%list_outlier$ID))
  assign(paste0("topo_", pair, "_cp_filtered_no_outliers"), new_topo_cp)
# write_csv(new_topo_cp,paste0("Topologies_all_topo_no_outliers/",pair,"_topo_limits.csv"))
  plots_hist <- c(plots_hist,plot_hist)
}

plots_hist

#### 2. Artemia 

In [ ]:
# Compute p-values for all genes in sin_urm pair
topo_sin_urm_cp_outliers <- filter(topo_sin_urm_cp, Start < 88970358) %>% 
  rowwise() %>%
  mutate(
    obs = list(c_across(`1010`:`0110`)),
    bloc = find_bloc(Start, summary_prop_sin_urm),
    lim1 = summary_prop_sin_urm$lim1[bloc],
    lim2 = summary_prop_sin_urm$lim2[bloc],
    prob_bloc = list({
      p <- unlist(summary_prop_sin_urm$proportions[[bloc]])
      p <- p + c(0,0,0,0,0,0,0,0, 0.01,0.01,0.01,0.01, 0,0)
      p / sum(p)   # mieux que /1.04
    }),
    size = sum(unlist(obs)),
    pvalue_sin_urm = prob_bloc_function(size,prob_bloc,obs,500000),
    Lpvalue_sin_urm = log(pvalue_sin_urm+(1/500000))
  ) %>%
  ungroup() %>%
  select(
    Chromosome, ID, Start, Length, obs, prob_bloc, size,
    pvalue_sin_urm,Lpvalue_sin_urm
  )

seuil_artemia <- -5

hist_artemia <-ggplot(topo_sin_urm_cp_outliers, aes(x = Lpvalue_sin_urm)) +
    geom_histogram(bins = 50, fill = "grey70", color = "black") +
    geom_vline(xintercept = seuil_artemia, color = "darkblue", linetype = "dashed", linewidth = 1) +
    ggtitle("A.sinica-A.urmiana") +
    xlab("pvalue") +
    theme_bw()

plots_hist<- c(plots_hist,hist_artemia)

test_artemia <- filter(topo_sin_urm_cp_outliers, Lpvalue_sin_urm < seuil_artemia)

topo_sin_urm_cp_no_outliers <- topo_traitement_function_for_cp(topo_sin_urm,list_gen_sin_urm,prop_duplicate_sin,prop_duplicate_urm,donnees_gtf_artemia) %>% filter(Chromosome == "Chromosome_1") %>% filter(!(ID%in%test_artemia$ID)) %>% filter(Start < 88970358)

# write_csv(topo_sin_urm_cp_no_outliers,paste0("Topologies_all_topo_no_outliers/","sin_urm","_topo_limits.csv"))

plot_histo <- wrap_plots(plots_hist, ncol = 4) +
    plot_layout(guides = "collect") &
     theme(panel.background = element_rect(fill = NA), #theme(legend.position = "none") +
      plot.background = element_rect(fill = NA))

#### 3. Change point analysis

In [ ]:
%use bash 
export WOLFRAMSCRIPT_KERNELPATH="/Applications/Wolfram.app/Contents/MacOS/MathKernel" 
pairs=("sin_urm" "lat_dio" "lat_dic" "lat_heu" "lat_mar" "dio_dic" "dio_heu" "dio_mar" "dic_heu" "dic_mar" "heu_mar")

cd /PATH/Analyses
for pair in "${pairs[@]}"; do     caffeinate -i ./change_point_recursive.sh ./cp.wls Topologies_all_topo_no_outliers/ "${pair}"; done

for f in Topologies_all_topo_no_outliers/topo0_cutted_*.csv; do echo -e "NA\nNA,NA\nNA,NA" > "$f"; echo "  remplacé : $f" ; done
for pair in "${pairs[@]}"; do     bash process_topo.sh "${pair}" Topologies_all_topo_no_outliers/; done

### II.5.f Plots without outliers 

#### 1. Data importation 

In [ ]:
for (pair in pairs) {
  tmp <- read.csv(paste0("Topologies_all_topo_no_outliers/topo_cutted_",pair,"_errors_final.csv"),header=FALSE)
  tmp<- tmp[order(tmp[,1]), , drop = FALSE]
  tmp2 <- read.csv(paste0("Topologies_all_topo_no_outliers/",pair,"_topo_limits.csv"), check.names = FALSE)
 assign(paste0("topo_", pair,"_cp_filtered_no_outliers"), tmp2)
 assign(paste0("cp_topo_", pair,"_errors_final_no_outliers"), tmp)
 assign(paste0("cp_topo_", pair,"_final_no_outliers"), sort(unlist(as.numeric(read.csv(paste0("Topologies_all_topo_no_outliers/topo_cutted_",pair,"_final.csv"),header=FALSE)))))
}

#### 2. Plot Silene

Same principle as described above

In [ ]:
# values used for the figure 2 in the paper
Gap_start =71e6
Gap_end = 251e6

# values used for the supplementary figure SX in the paper
# Gap_start =101e6
# Gap_end = 226e6


plot_topo_silene_function_subset <- function(xlim1, xlim2, cp_topo_final, cp_topo_errors_final, topo_traited_filtered, b_list, gap_start = Gap_start, gap_end = Gap_end) {
  b_df <- process_blocs(b_list, gap_start, gap_end)
  x_breaks_real <- seq(from = ceiling(xlim1/2.5e7)*2.5e7,
                       to   = floor(xlim2/2.5e7)*2.5e7,
                       by   = 2.5e7)
  x_breaks_real <- x_breaks_real[!(x_breaks_real > gap_start & x_breaks_real < gap_end)]
  x_breaks_compressed <- compress_x(x_breaks_real)
  x_labels <- scales::scientific(x_breaks_real)
  plot <- ggplot(topo_traited_filtered) +
    geom_rect(
      data = b_df,
      aes(xmin = lim1, xmax = lim2, ymin = cum_low, ymax = cum_high, fill = variable),
      color = NA, alpha = 0.3, inherit.aes = FALSE
    ) +
    geom_vline(xintercept = compress_x(cp_topo_final), linetype = "dashed", color = "darkgray") +
    geom_rect(
      data = cp_topo_errors_final,
      aes(xmin = compress_x(V1), xmax = compress_x(V2), ymin = -Inf, ymax = Inf),
      inherit.aes = FALSE, fill = "darkgray", alpha = 0.2
    ) +
     scale_fill_manual(values = c( "#FFCC02", "#FF6D00", "#B600FF" , "#A9D400" , "#03FFFF","#4D4D4D","#784321" ,"#FF0000","black","#0024FF","#DE8787" , "#FF00DB" ,"#008001", "#019E9E")) +   
    theme_bw() +
    theme(legend.position = "none") +
    labs(x = "Position (bp)",y = "Topologies proportion")+
    facet_grid(vars(Pair)) +
    scale_x_continuous(breaks = x_breaks_compressed, labels = x_labels, expand = c(0, 0)) +
    coord_cartesian(xlim = compress_x(c(xlim1, xlim2))) +
    annotate("segment", x = gap_start - 1e6, xend = gap_start + 1e6, y = -0.03, yend = -0.01, size = 0.8) +
    annotate("segment", x = gap_start - 1e6, xend = gap_start + 1e6, y = -0.01, yend = -0.03, size = 0.8)
  summary(b_df$lim1); summary(b_df$lim2)
  return(plot)
}

combined_topo_plot_silene_function_subset <- function(xlim1, xlim2) {
  
  plot_list <- list()
  for(i in seq_along(pairs[-1])) {
    pair <- pairs[-1][i]
    topo_cp <- get(paste0("topo_", pair, "_cp_filtered_no_outliers"))
    topo_traited_filtered <- get(paste0("topo_", pair, "_traited_filtered"))
    cp_topo_final <-get(paste0("cp_topo_", pair,"_final_no_outliers"))
    cp_topo_errors_final <- get(paste0("cp_topo_", pair, "_errors_final_no_outliers"))
    cp <- sort(cp_topo_final)
    starts <- c(0, cp)
    ends <- c(cp, last(sort(topo_traited_filtered$Start))+1)
    
    b_list <- Map(function(s,e) {
      bloc <- prop_function(topo_cp, s, e, pair)
      bloc$lim1 <- bloc$lim1
      bloc$lim2 <- bloc$lim2
      bloc
    }, starts, ends)
    
    summary_table <- tibble(
      pair = pair,
      bloc = seq_along(b_list),
      lim1 = map_dbl(b_list, ~ unique(.x$lim1)),
      lim2 = map_dbl(b_list, ~ unique(.x$lim2)),
      proportions = map(b_list, ~ .x$proportion)
    )
    assign(paste0("summary_prop_", pair,"_no_outliers"), summary_table, envir = .GlobalEnv)
    
    plot <- plot_topo_silene_function_subset(
      xlim1 = xlim1,
      xlim2 = xlim2,
      cp_topo_final = cp_topo_final,
      cp_topo_errors_final = cp_topo_errors_final,
      topo_traited_filtered = topo_traited_filtered,
      b_list = b_list
    ) 
    
    if(i < length(pairs[-1])) {
      plot <- plot + theme(axis.title.x = element_blank(),
                           axis.text.x  = element_blank(),
                           axis.ticks.x = element_blank())
    } else {
      plot <- plot +  geom_point(data = arrows,
             aes(x = compress_x(x), y = y),  
             shape = 17, size = 2.5,
             color = "blue",
             inherit.aes = FALSE)
    }
    
    plot_list[[pair]] <- plot 
    
  }
  
  plot_zoom <- wrap_plots(plot_list, ncol = 1) +
    plot_layout(guides = "collect") &
     theme(panel.background = element_rect(fill = NA), 
      plot.background = element_rect(fill = NA))
  
  return(plot_zoom)
}

plot_silene_total_no_outliers <- combined_topo_plot_silene_function_subset(0, 340000000)
plot_silene_total_no_outliers

In [ ]:
summary_bloc_silene <- data.frame()
for(i in seq_along(pairs)) {
    pair <- pairs[i]
    data<- get(paste0("summary_prop_", pair,"_no_outliers"))
    data <-data %>% 
    unnest(proportions) %>% 
    mutate(proportions = round(proportions, 2)) %>%
    group_by(bloc) %>%
    mutate(position = row_number()) %>%
    ungroup() %>%
    pivot_wider(
      id_cols = c(pair, bloc, lim1, lim2),
      names_from = position,
      values_from = proportions
    ) %>%
    rename_with(~desired_order[as.numeric(.)], matches("^[0-9]+$"))
    summary_bloc_silene <- rbind(summary_bloc_silene,data)
}

In [ ]:
write.csv(summary_bloc_silene, "summary_bloc_silene.csv", row.names = FALSE)

#### 3. Plot Artemia 

In [ ]:
topo_sin_urm_cp_no_outliers <- read.csv("Topologies_all_topo_no_outliers/sin_urm_topo_limits.csv", check.names = FALSE)
cp_artemia_no_outliers <- sort(cp_topo_sin_urm_final_no_outliers)
starts_A_no_outliers <- c(first(sort(topo_sin_urm_cp$Start)), cp_artemia_no_outliers)
ends_A_no_outliers <- c(cp_artemia_no_outliers, 88970358+1) 
    
b_list_artemia_no_outliers <- Map(function(s,e) {
  bloc <- prop_function(filter(topo_sin_urm_cp_no_outliers, Start <88970358), s, e, pair)
  bloc$lim1 <- bloc$lim1
  bloc$lim2 <- bloc$lim2
  bloc
}, starts_A_no_outliers, ends_A_no_outliers)

summary_prop_sin_urm_no_outliers <- tibble(
  pair = "sin_urm",
  bloc = seq_along(b_list_artemia_no_outliers),
  lim1 = map_dbl(b_list_artemia_no_outliers, ~ unique(.x$lim1)),
  lim2 = map_dbl(b_list_artemia_no_outliers, ~ unique(.x$lim2)),
  proportions = map(b_list_artemia_no_outliers, ~ .x$proportion))
    
# Plot
plot_artemia_total_no_outliers <- plot_artemia_function_subset(
  xlim1 = 0,
  xlim2 = 350000000,
  cp_topo_final = cp_topo_sin_urm_final_no_outliers,
  cp_topo_errors_final = cp_topo_sin_urm_errors_final_no_outliers,
  topo_traited = topo_sin_urm_traited,
  b_list = b_list_artemia_no_outliers
) + scale_x_continuous(limits = c(0, 100800000)) + 
  geom_point(data = arrows_artemia, aes(x = x, y = y),  # yend = hauteur où tu veux le triangle
         shape = 17, size = 2,
         color = "blue",
         inherit.aes = FALSE) 

plot_artemia_total_no_outliers

In [ ]:
summary_prop_sin_urm_no_outliers

## II.6. Codes for specific plots of the paper

### II.6.a On gene G

In [ ]:
sin_urm_full<-topo_importation_function("topo/topo_sin_urm_full.csv")

g1385_for_plot <- sin_urm_full %>% filter(ID=="g1385.t1") %>% relocate(all_of(desired_order), .after = "ID") %>% melt(id.vars = c("ID", "position")) %>% filter(value==1) %>% filter(variable!="1111")

boundaries <- donnees_gtf_artemia%>%filter(gene_ID=="g1385") %>% filter(Type=="exon") %>% select(Start,End) %>% mutate(length = End -Start +1, border_2 = cumsum(length), border_1 = border_2 -length ) 

segment_colors <- c("gray","black")
g1385_plot <- ggplot(g1385_for_plot, aes(x = position, y = 0, color = variable)) +
  annotate("segment",x=1,xend=1590, y=0, yend=0, size=0.5) +
  
  geom_segment(
    data = boundaries,
    aes(x = border_1, xend = border_2, y = -0.3, yend = -0.3),
    color = segment_colors[1:nrow(boundaries) %% 2 + 1],  # Alternance noir/gris
    size = 1
  ) +
  
  annotate("segment",x=888,xend=888, y=-0.35,yend=-0.25, size=1,color="black") +
  annotate("segment",x=932,xend=932, y=-0.35,yend=-0.25, size=1,color="black") +
  annotate("segment",x=0,xend=0, y=-0.35,yend=-0.25, size=1,color="black") +
  annotate("segment",x=1590,xend=1590, y=-0.35,yend=-0.25, size=1,color="black") +
  scale_y_continuous(limits = c(-1,1)) +
  geom_point(size = 3) +

  scale_colour_manual(values = c( "#0024FF","#03FFFF", "#A9D400","#FFCC02","#FF0000", "#FF6D00", "#B600FF" , "#FF00DB" ,"#019E9E"  ,"#008001" , "#DE8787","#784321","#4D4D4D","black")) +
  theme(panel.background = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        axis.title = element_blank(),
        legend.position = "none")


### II.6.b Comparision with and without pop data for *Silene latifolia*

In [ ]:
plot_dS_lat_dio<- ggplot(dS_merged_silene %>% filter(Species==c("S.latifolia","S.dioica")), aes(x = Middle, y = dS)) +
  geom_point(size = 1) +
  xlab("position (pdb)") +
  ylab("dS") +
  #geom_vline(xintercept = c(48638, 5769054, 10069580, 15830480, 36776284), 
             #linetype = "dashed", color = "blue") +
  
  facet_grid(vars(Species)) +
  #facet_grid(Species ~ ., scales = "free_x") +
  scale_x_continuous(limits = c(300000000, 350000000),expand = c(0, 0)) +
  ylim(0, 0.2) +
  theme(legend.position = "none") +
  theme_bw() #+

plot_dS_lat_dio_no_filter<- ggplot(dS_merged_silene_no_filter %>% filter(Species==c("S.latifolia","S.dioica")), aes(x = Middle, y = dS)) +
  geom_point(size = 1) +
  xlab("position (pdb)") +
  ylab("dS") +
  #geom_vline(xintercept = c(48638, 5769054, 10069580, 15830480, 36776284), 
             #linetype = "dashed", color = "blue") +
  facet_grid(Species ~ .) +
  scale_x_continuous(limits = c(300000000, 350000000),expand = c(0, 0)) +
  ylim(0, 0.2) +
  theme_bw() +
  theme(legend.position = "none") 
lat_dio_par<-plot_topo_function(300000000/1e6,350000000/1e6,topo_lat_dio_traited,topo_lat_dio_smooth)

lat_dio_par_filtered<-plot_topo_function(300000000/1e6,350000000/1e6,topo_lat_dio_traited_filtered,topo_lat_dio_smooth_filtered)


plot_pop_clean_filtered <- wrap_plots(c(plot_dS_lat_dio + theme(axis.title.x = element_blank(),
                           axis.text.x  = element_blank(),
                           axis.ticks.x = element_blank()),lat_dio_par_filtered), ncol = 1) +
plot_layout(guides = "collect") & theme(legend.position = "none") + theme(panel.background = element_rect(fill = NA),
plot.background = element_rect(fill = NA))

plot_pop_clean <- wrap_plots(c(plot_dS_lat_dio_no_filter + theme(axis.title.x = element_blank(),
                           axis.text.x  = element_blank(),
                           axis.ticks.x = element_blank()),lat_dio_par), ncol = 1) +
plot_layout(guides = "collect") & theme(legend.position = "none") + theme(panel.background = element_rect(fill = NA),
plot.background = element_rect(fill = NA))

plot_pop_clean_total <- wrap_plots(c(plot_pop_clean,plot_pop_clean_filtered), ncol=2)


## II.7 Saving plots 

### II.7.a Artemia dS and topologies

In [ ]:
plot_artemia_general <- wrap_plots(c(plot_dS_artemia_total,plot_artemia_total_no_outliers), ncol = 1) +
plot_layout(guides = "collect") & theme(legend.position = "none") + theme(panel.background = element_rect(fill = NA),
plot.background = element_rect(fill = NA))
plot_artemia_general

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/Reedaction/Figures/plot_general_artemia.pdf",
    width = 8, # The width of the plot in inches
    height = 5) # The height of the plot in inches
plot_artemia_general
dev.off()

In [ ]:
plot_artemia_general_outliers <- wrap_plots(c(plot_dS_artemia_total,plot_artemia_total), ncol = 1) +
plot_layout(guides = "collect") & theme(legend.position = "none") + theme(panel.background = element_rect(fill = NA),
plot.background = element_rect(fill = NA))
plot_artemia_general_outliers

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/Rédaction/Figures/plot_general_artemia_outliers.pdf",
    width = 8, # The width of the plot in inches
    height = 5) # The height of the plot in inches
plot_artemia_general_outliers
dev.off()

### II.7.b Silene dS

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/Redaction/Figures/plot_dS_silene.pdf",
    width = 8, # The width of the plot in inches
    height = 6.5) # The height of the plot in inches
plot_dS_silene_total
dev.off()

### II.7.c Silene topologies

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/PAPIER_METHODE/Redaction/Figures/plot_topo_silene.pdf",
    width = 8, # The width of the plot in inches
    height = 9) # The height of the plot in inches
plot_silene_total_no_outliers
dev.off()

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/PAPIER_METHODE/Redaction/Figures/plot_topo_silene_outlier.pdf",
    width = 8, # The width of the plot in inches
    height = 9) # The height of the plot in inches
plot_silene_total
dev.off()

### II.7.d Outliers histograms 

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/Redaction/Figures/plot_histo.pdf",
    width = 10, # The width of the plot in inches
    height = 8) # The height of the plot in inches
plot_histo
dev.off()

### II.7.e On gene G

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/Rédaction/Figures/plot_g1385.pdf",
    width = 8, # The width of the plot in inches
    height = 3) # The height of the plot in inches
g1385_plot
dev.off()

### II.7.f Comparision with and without pop data for *Silene latifolia*

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/Rédaction/Figures/plot_clean_pop.pdf",
    width = 9, # The width of the plot in inches
    height = 5) # The height of the plot in inches
plot_pop_clean_total
dev.off()

### II.7.g Sex-linked genes

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/Rédaction/Figures/plot_SEX_DET_silene.pdf",
    width = 10, # The width of the plot in inches
    height = 5) # The height of the plot in inches
P1
dev.off()

In [ ]:

pdf(file = "~/Documents/THESE/GENOMIQUE/Rédaction/Figures/plot_SEX_DET_artemia.pdf",
    width = 10, # The width of the plot in inches
    height = 2.75) # The height of the plot in inches
P2
dev.off()

### II.7.h Supplementary figure : ds and topologies for silene in one figure

In [ ]:
#before doing this make sure to have the same gap start and end (101 and 226 respectivly) in the two separate plots (plot_dS_silene_total and plot_silene_total_no_outliers) and the same xlim for both plots

plot_silene_general <- wrap_plots(c(plot_dS_silene_total + theme(axis.text.x = element_blank(), axis.ticks.x = element_blank(), axis.title.x = element_blank()),plot_silene_total_no_outliers), ncol = 1, heights = c(1, 2)) +
plot_layout(guides = "collect") & theme(legend.position = "none") + theme(panel.background = element_rect(fill = NA),
plot.background = element_rect(fill = NA))
plot_silene_general

In [ ]:
pdf(file = "~/Documents/THESE/GENOMIQUE/PAPIER_METHODE/Redaction/Figures/plot_general_silene.pdf",
    width = 8, # The width of the plot in inches
    height = 13) # The height of the plot in inches
plot_silene_general
dev.off()